### Figure 4 — Input and output directories


In [ ]:
NOTEBOOK_NAME = 'fig4'
from pathlib import Path
import os
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / 'pipelines/run.py').is_file()))
INPUT_ROOT = Path(os.environ.get('CIPHER_ANALYSIS_ROOT', str(ROOT / 'figure_data'))).resolve()
OUTPUT_ROOT = ROOT / 'figure_outputs' / NOTEBOOK_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_ROOT)


### Figure 4b — Check prediction-table coverage


In [ ]:
import pandas as pd
import glob
import os
data_dir = 'code_02_02_save/snp_20260202_1432/preds'
pattern = os.path.join(data_dir, '*_test_best_pred.csv')
file_list = glob.glob(str(OUTPUT_ROOT / pattern if (OUTPUT_ROOT / pattern).exists() else INPUT_ROOT / pattern))
if not file_list:
    print('No *_pred.csv files found. Check the configured input directory.')
else:
    all_ids = set()
    for file_path in file_list:
        try:
            df = pd.read_csv(str(OUTPUT_ROOT / file_path if (OUTPUT_ROOT / file_path).exists() else INPUT_ROOT / file_path), usecols=['ID'])
            all_ids.update(df['ID'].tolist())
        except KeyError:
            df = pd.read_csv(str(OUTPUT_ROOT / file_path if (OUTPUT_ROOT / file_path).exists() else INPUT_ROOT / file_path), header=0)
            first_col = df.columns[0]
            all_ids.update(df[first_col].tolist())
        except Exception as e:
            print(f'Error reading {file_path}: {e}')
    print(f'Number of unique participants across endpoints:{len(all_ids)}')


### Figure 4b — Modality-combination performance


In [ ]:
from sklearn.metrics import roc_auc_score
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines.utils import concordance_index
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE_DIR = 'fig3/visit02'
OUT_CINDEX = 'fig4_final/fig4_visit02_cindex_barplot_by_combo.pdf'
OUT_EVENT_AUROC = 'fig4_final/fig4_visit02_event_auroc_barplot_by_combo.pdf'
ENDPOINT_ORDER = ['Coronary Heart Disease', 'Ischemic Stroke', 'ASCVD']
ENDPOINT_ORDER = ['Ischemic Stroke', 'ASCVD']
ID_COL = 'ID'
sns.set_style('white')
ONE_MOD_COLORS = {'ecg': '#E6D2BD', 'snp': '#D9C6A9', 'soma': '#CDB48F', 'fundus': '#BFA27A'}
TWO_MOD_PALETTE = ['#C9D7E6', '#B8CBE0', '#A7BEDA', '#96B2D4', '#85A5CE', '#7499C8']
THREE_MOD_PALETTE = ['#CFE0C8', '#BFD6B6', '#AFCDA4', '#9FC392']
FOUR_MOD_COLOR = '#8F3B3B'

def center_only_normalize(x):
    x = np.asarray(x)
    return x - np.mean(x)

def compute_cindex(time, event, risk):
    return concordance_index(time, -risk, event)

def binary_event_auroc(time, event, risk):
    return float(roc_auc_score(event, risk)) if np.unique(event).size == 2 else float('nan')

def parse_combo(folder):
    parts = folder.split('_')
    cut = None
    for (i, p) in enumerate(parts):
        if re.fullmatch('\\d{8}', p):
            cut = i
            break
    if cut is None:
        cut = len(parts)
    combo_parts = parts[:cut]
    combo = '+'.join(combo_parts)
    return (combo_parts, combo)

def load_pred(path):
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    return df[[ID_COL, 'time', 'event', 'risk']].copy()
runs = []
for folder in os.listdir(BASE_DIR):
    if 'best' in folder:
        continue
    fpath = os.path.join(BASE_DIR, folder)
    if not os.path.isdir(str(OUTPUT_ROOT / fpath if (OUTPUT_ROOT / fpath).exists() else INPUT_ROOT / fpath)):
        continue
    (combo_parts, combo) = parse_combo(folder)
    n_mod = len(combo_parts)
    files = {'Ischemic Stroke': os.path.join(fpath, 'istroke_pred.csv'), 'ASCVD': os.path.join(fpath, 'ICVD_pred.csv')}
    if not all((os.path.exists(str(OUTPUT_ROOT / x if (OUTPUT_ROOT / x).exists() else INPUT_ROOT / x)) for x in files.values())):
        continue
    for (endpoint, path) in files.items():
        df = load_pred(path)
        time = df['time'].values
        event = df['event'].astype(bool).values
        risk = center_only_normalize(df['risk'].values)
        cindex = compute_cindex(time, event, risk)
        event_auroc = binary_event_auroc(time, event, risk)
runs = pd.DataFrame(runs)
rows = []
for ((combo, endpoint), sub) in runs.groupby(['combo', 'endpoint']):
    sub = sub.sort_values('cindex', ascending=False)
dfm = pd.DataFrame(rows)

def combo_key(c):
    parts = c.split('+')
    return (len(parts), c)
combo_order = sorted(dfm['combo'].unique(), key=combo_key)
combo_color = {}
for c in combo_order:
    n = len(c.split('+'))
    if n == 1:
        combo_color[c] = ONE_MOD_COLORS.get(c, '#CCCCCC')
    elif n == 2:
        combo_color[c] = TWO_MOD_PALETTE[combo_order.index(c) % len(TWO_MOD_PALETTE)]
    elif n == 3:
        combo_color[c] = THREE_MOD_PALETTE[combo_order.index(c) % len(THREE_MOD_PALETTE)]
    else:
        combo_color[c] = FOUR_MOD_COLOR

def plot_metric(mean_col, sd_col, label, outfile):
    (fig, ax) = plt.subplots(figsize=(10, 3))
    x_pos = []
    heights = []
    errors = []
    colors = []
    xticks = []
    xticklabels = []
    width = 0.8 / len(combo_order)
    for (i, endpoint) in enumerate(ENDPOINT_ORDER):
        sub = dfm[dfm.endpoint == endpoint]
        for (j, combo) in enumerate(combo_order):
            row = sub[sub.combo == combo]
            if len(row) == 0:
                continue
            mean = row[mean_col].values[0]
            sd = row[sd_col].values[0]
            x = i + j * width
            x_pos.append(x)
            heights.append(mean)
            errors.append(sd)
            colors.append(combo_color[combo])
        xticks.append(i + width * len(combo_order) / 2)
    ax.bar(x_pos, heights, width=width, color=colors, edgecolor='black', linewidth=0.4, yerr=errors, capsize=3)
    ax.set_xticks(xticks)
    ax.set_xticklabels(ENDPOINT_ORDER)
    ax.set_ylabel(label)
    ax.set_ylim(0.45, 0.75)
    ax.yaxis.grid(True, linestyle='--', alpha=0.35)
    sns.despine()
    plt.tight_layout()
    plt.savefig(outfile)
    print('Saved:', outfile)
plot_metric('cindex_mean', 'cindex_sd', 'C-index', OUT_CINDEX)
plot_metric('event_auroc_mean', 'event_auroc_sd', 'Event-status AUROC', OUT_EVENT_AUROC)
from matplotlib.patches import Patch
handles = [Patch(facecolor=combo_color[c], edgecolor='black', label=c) for c in combo_order]
(fig, ax) = plt.subplots(figsize=(10, 3))
ax.legend(handles=handles, ncol=5, frameon=False, title='Omics combo (fewer → more)', loc='center')
ax.axis('off')
plt.savefig('fig4_final/fig4_visit02_legend.pdf', bbox_inches='tight')
print('\n======================================')
print('BEST MODELS PER DISEASE')
print('======================================')
for endpoint in ENDPOINT_ORDER:
    sub = runs[(runs['endpoint'] == endpoint) & (runs['combo'] == 'ecg+soma+snp+fundus')].copy()
    best_c = sub.sort_values('cindex', ascending=False).iloc[0]
    best_auc = sub.sort_values('event_auroc', ascending=False).iloc[0]
    print('\n----------------------------------')
    print(endpoint)
    print('----------------------------------')
    print('\nBest C-index model:')
    print('Combo :', best_c['combo'])
    print('Folder:', best_c['folder'])
    print('C-index:', round(best_c['cindex'], 4))
    print('AUC:', round(best_c['event_auroc'], 4))
    print('\nBest AUC model:')
    print('Combo :', best_auc['combo'])
    print('Folder:', best_auc['folder'])
    print('C-index:', round(best_auc['cindex'], 4))
    print('AUC:', round(best_auc['event_auroc'], 4))


### Figure 4b — Longitudinal-run comparison


In [ ]:
from sklearn.metrics import roc_auc_score
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines.utils import concordance_index
BASE_DIR = 'fig3/longitudinal'
OUT_CINDEX = 'fig4_final/fig4_longitudinal_cindex_barplot_by_combo.pdf'
OUT_EVENT_AUROC = 'fig4_final/fig4_longitudinal_event_auroc_barplot_by_combo.pdf'
ENDPOINT_ORDER = ['Coronary Heart Disease', 'Ischemic Stroke', 'ASCVD']
ID_COL = 'ID'
sns.set_style('white')
ONE_MOD_COLORS = {'ecg': '#E6D2BD', 'snp': '#D9C6A9', 'soma': '#CDB48F', 'fundus': '#BFA27A'}
TWO_MOD_PALETTE = ['#C9D7E6', '#B8CBE0', '#A7BEDA', '#96B2D4', '#85A5CE', '#7499C8']
THREE_MOD_PALETTE = ['#CFE0C8', '#BFD6B6', '#AFCDA4', '#9FC392']
FOUR_MOD_COLOR = '#8F3B3B'

def center_only_normalize(x):
    x = np.asarray(x)
    return x - np.mean(x)

def compute_cindex(time, event, risk):
    return concordance_index(time, -risk, event)

def binary_event_auroc(time, event, risk):
    return float(roc_auc_score(event, risk)) if np.unique(event).size == 2 else float('nan')

def parse_combo(folder):
    parts = folder.split('_')
    cut = None
    for (i, p) in enumerate(parts):
        if re.fullmatch('\\d{8}', p):
            cut = i
            break
    if cut is None:
        cut = len(parts)
    combo_parts = parts[:cut]
    combo = '+'.join(combo_parts)
    return (combo_parts, combo)

def load_pred(path):
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    return df[[ID_COL, 'time', 'event', 'risk']].copy()
runs = []
for folder in os.listdir(BASE_DIR):
    if 'best' in folder:
        continue
    fpath = os.path.join(BASE_DIR, folder)
    if not os.path.isdir(str(OUTPUT_ROOT / fpath if (OUTPUT_ROOT / fpath).exists() else INPUT_ROOT / fpath)):
        continue
    (combo_parts, combo) = parse_combo(folder)
    n_mod = len(combo_parts)
    files = {'Coronary Heart Disease': os.path.join(fpath, 'CHD_pred.csv'), 'Ischemic Stroke': os.path.join(fpath, 'istroke_pred.csv'), 'ASCVD': os.path.join(fpath, 'ICVD_pred.csv')}
    if not all((os.path.exists(str(OUTPUT_ROOT / x if (OUTPUT_ROOT / x).exists() else INPUT_ROOT / x)) for x in files.values())):
        continue
    for (endpoint, path) in files.items():
        df = load_pred(path)
        time = df['time'].values
        event = df['event'].astype(bool).values
        risk = center_only_normalize(df['risk'].values)
        cindex = compute_cindex(time, event, risk)
        event_auroc = binary_event_auroc(time, event, risk)
runs = pd.DataFrame(runs)
rows = []
for ((combo, endpoint), sub) in runs.groupby(['combo', 'endpoint']):
    sub = sub.sort_values('cindex', ascending=False)
dfm = pd.DataFrame(rows)

def combo_key(c):
    parts = c.split('+')
    return (len(parts), c)
combo_order = sorted(dfm['combo'].unique(), key=combo_key)
combo_color = {}
for c in combo_order:
    n = len(c.split('+'))
    if n == 1:
        combo_color[c] = ONE_MOD_COLORS.get(c, '#CCCCCC')
    elif n == 2:
        combo_color[c] = TWO_MOD_PALETTE[combo_order.index(c) % len(TWO_MOD_PALETTE)]
    elif n == 3:
        combo_color[c] = THREE_MOD_PALETTE[combo_order.index(c) % len(THREE_MOD_PALETTE)]
    else:
        combo_color[c] = FOUR_MOD_COLOR

def plot_metric(mean_col, sd_col, label, outfile):
    (fig, ax) = plt.subplots(figsize=(12.5, 2.3))
    x_pos = []
    heights = []
    errors = []
    colors = []
    xticks = []
    xticklabels = []
    width = 0.8 / len(combo_order)
    for (i, endpoint) in enumerate(ENDPOINT_ORDER):
        sub = dfm[dfm.endpoint == endpoint]
        for (j, combo) in enumerate(combo_order):
            row = sub[sub.combo == combo]
            if len(row) == 0:
                continue
            mean = row[mean_col].values[0]
            sd = row[sd_col].values[0]
            x = i + j * width
            x_pos.append(x)
            heights.append(mean)
            errors.append(sd)
            colors.append(combo_color[combo])
        xticks.append(i + width * len(combo_order) / 2)
    ax.bar(x_pos, heights, width=width, color=colors, edgecolor='black', linewidth=0.4, yerr=errors, capsize=3)
    ax.set_xticks(xticks)
    ax.set_xticklabels(ENDPOINT_ORDER)
    ax.set_ylabel(label)
    ax.set_ylim(0.4, 0.85)
    ax.yaxis.grid(True, linestyle='--', alpha=0.35)
    sns.despine()
    plt.tight_layout()
    plt.savefig(outfile)
    print('Saved:', outfile)
plot_metric('cindex_mean', 'cindex_sd', 'C-index', OUT_CINDEX)
plot_metric('event_auroc_mean', 'event_auroc_sd', 'Event-status AUROC', OUT_EVENT_AUROC)
from matplotlib.patches import Patch
handles = [Patch(facecolor=combo_color[c], edgecolor='black', label=c) for c in combo_order]
(fig, ax) = plt.subplots(figsize=(10, 1.6))
ax.legend(handles=handles, ncol=5, frameon=False, title='Omics combo (fewer → more)', loc='center')
ax.axis('off')
plt.savefig('fig4_final/fig3_longitudinal_legend.pdf', bbox_inches='tight')


### Figure 4b — Four-modality model results


In [ ]:
print('\n======================================')
print('BEST MODELS PER DISEASE')
print('======================================')
for endpoint in ENDPOINT_ORDER:
    sub = runs[(runs['endpoint'] == endpoint) & (runs['combo'] == 'ecg+soma+snp+fundus')].copy()
    best_c = sub.sort_values('cindex', ascending=False).iloc[0]
    best_auc = sub.sort_values('event_auroc', ascending=False).iloc[0]
    print('\n----------------------------------')
    print(endpoint)
    print('----------------------------------')
    print('\nBest C-index model:')
    print('Combo :', best_c['combo'])
    print('Folder:', best_c['folder'])
    print('C-index:', round(best_c['cindex'], 4))
    print('AUC:', round(best_c['event_auroc'], 4))
    print('\nBest AUC model:')
    print('Combo :', best_auc['combo'])
    print('Folder:', best_auc['folder'])
    print('C-index:', round(best_auc['cindex'], 4))
    print('AUC:', round(best_auc['event_auroc'], 4))


### Figure 4c — Multimodal integration strategies, separate plots


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE = '.'
IN_DIR = os.path.join(BASE, 'code_3_3_save/compare_external_mean_fivefold_fullcox_visit02')
SUMMARY_CSV = os.path.join(IN_DIR, 'external_comparison_cindex_auc_summary.csv')
OUT_DIR = os.path.join('fig4_final', 'figures_visit02_style_istroke_ascvd')
os.makedirs(OUT_DIR, exist_ok=True)
COLOR_VISIT02 = '#8B1E1E'
COLOR_OTHER = '#A3A9A6'
TARGET_ENDPOINTS = ['istroke', 'ICVD']
ENDPOINT_DISPLAY = {'istroke': 'Ischemic stroke', 'ICVD': 'ASCVD'}
VISIT02_METHOD_RAW = 'Visit02 best multimodal'
METHOD_LABELS = {'External direct mean': 'Direct mean', 'Fivefold Cox weighted': '5-fold Cox weighted', 'Full-train Cox weighted': 'Full-train Cox weighted', 'Visit02 best multimodal': 'latent-based multimodel aggregate'}
OTHER_METHOD_ORDER = ['External direct mean', 'Fivefold Cox weighted', 'Full-train Cox weighted']

def load_summary(summary_csv):
    if not os.path.exists(str(OUTPUT_ROOT / summary_csv if (OUTPUT_ROOT / summary_csv).exists() else INPUT_ROOT / summary_csv)):
        raise FileNotFoundError(f'Cannot find summary CSV:\n{summary_csv}\n\nRun the upstream analysis to generate external_comparison_cindex_auc_summary.csv.')
    df = pd.read_csv(str(OUTPUT_ROOT / summary_csv if (OUTPUT_ROOT / summary_csv).exists() else INPUT_ROOT / summary_csv)).rename(columns={'n': 'n_analyzed'})
    if 'event_auroc' not in df.columns and 'auc' in df.columns:
        df['event_auroc'] = df['auc']
    required = ['endpoint', 'method', 'n_analyzed', 'n_event', 'cindex', 'event_auroc']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'summary CSV missing columns: {missing}')
    df = df.copy()
    df = df[df['endpoint'].isin(TARGET_ENDPOINTS)].copy()
    if len(df) == 0:
        raise ValueError('No istroke / ICVD rows found in summary CSV. Please check endpoint names.')
    df['endpoint_display'] = df['endpoint'].map(ENDPOINT_DISPLAY)
    df['method_display'] = df['method'].map(METHOD_LABELS)
    df['method_display'] = df['method_display'].fillna(df['method'])
    return df

def plot_visit02_style(summary, metric='cindex', out_pdf=None, out_png=None, figsize=(6.6, 4.4)):
    if metric not in ['cindex', 'event_auroc']:
        raise ValueError("metric must be 'cindex' or 'event_auroc'")
    endpoints = [ep for ep in TARGET_ENDPOINTS if ep in summary['endpoint'].unique()]
    n_panels = len(endpoints)
    (fig, axes) = plt.subplots(n_panels, 1, figsize=figsize, sharex=False)
    if n_panels == 1:
        axes = [axes]
    for (ax, endpoint) in zip(axes, endpoints):
        sub = summary[summary['endpoint'] == endpoint].copy()
        sub[metric] = pd.to_numeric(sub[metric], errors='coerce')
        sub = sub.dropna(subset=[metric]).copy()
        if len(sub) == 0:
            ax.text(0.5, 0.5, 'No valid result', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(ENDPOINT_DISPLAY.get(endpoint, endpoint))
            continue
        visit02_sub = sub[sub['method'] == VISIT02_METHOD_RAW].copy()
        if len(visit02_sub) == 0:
            raise ValueError(f"{endpoint}: cannot find method '{VISIT02_METHOD_RAW}' in summary CSV.")
        visit02_val = float(visit02_sub[metric].iloc[0])
        other = sub[sub['method'] != VISIT02_METHOD_RAW].copy()
        ordered_other_rows = []
        for m in OTHER_METHOD_ORDER:
            tmp = other[other['method'] == m].copy()
            if len(tmp) > 0:
                ordered_other_rows.append(tmp.iloc[[0]])
        remaining = other[~other['method'].isin(OTHER_METHOD_ORDER)].copy()
        if len(remaining) > 0:
            remaining = remaining.sort_values(metric, ascending=True)
            ordered_other_rows.append(remaining)
        if len(ordered_other_rows) > 0:
            other_ordered = pd.concat(ordered_other_rows, axis=0, ignore_index=True)
        else:
            other_ordered = pd.DataFrame(columns=sub.columns)
        plot_df = pd.concat([other_ordered, visit02_sub.iloc[[0]]], axis=0, ignore_index=True)
        y = np.arange(len(plot_df))
        values = plot_df[metric].astype(float).values
        labels = plot_df['method_display'].tolist()
        methods = plot_df['method'].tolist()
        colors = [COLOR_VISIT02 if m == VISIT02_METHOD_RAW else COLOR_OTHER for m in methods]
        ax.barh(y, values, color=colors, height=0.8)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=10)
        ax.invert_yaxis()
        ax.axvline(visit02_val, linestyle='--', color=COLOR_VISIT02, linewidth=1.5, alpha=0.9)
        for (yy, val, method) in zip(y, values, methods):
            if np.isnan(val):
                continue
            if method == VISIT02_METHOD_RAW:
                ax.text(val + 0.004, yy, f'{val:.3f}', va='center', ha='left', fontsize=10, color=COLOR_VISIT02)
            else:
                ax.text(val + 0.004, yy, f'{val:.3f}', va='center', ha='left', fontsize=9, color='black', alpha=0.75)
        for (yy, val, method) in zip(y, values, methods):
            if method == VISIT02_METHOD_RAW:
                continue
            if np.isnan(val) or val == 0:
                continue
            gain = (visit02_val - val) / val * 100
            sign = '+' if gain >= 0 else ''
            gain_text = f'{sign}{gain:.1f}%'
            ax.text(visit02_val + 0.055, yy, gain_text, va='center', ha='left', fontsize=10.5, color=COLOR_VISIT02)
        n_val = int(visit02_sub['n_analyzed'].iloc[0])
        e_val = int(visit02_sub['n_event'].iloc[0])
        ax.set_title(f'{ENDPOINT_DISPLAY.get(endpoint, endpoint)}  (analyzed n={n_val}, events={e_val})', fontsize=13, pad=8)
        if metric == 'cindex':
            ax.set_xlabel('C-index', fontsize=11)
        else:
            ax.set_xlabel('Event AUROC', fontsize=11)
        ax.grid(axis='x', linestyle='--', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        xmin = max(0.45, np.nanmin(values) - 0.03)
        xmax = min(1.02, max(np.nanmax(values) + 0.16, visit02_val + 0.16))
        ax.set_xlim(0.6, 0.75)
    plt.tight_layout()
    if out_pdf is not None:
        plt.savefig(out_pdf, transparent=True, bbox_inches='tight')
        print('Saved:', out_pdf)
    if out_png is not None:
        plt.savefig(out_png, dpi=300, transparent=True, bbox_inches='tight')
        print('Saved:', out_png)
    plt.show()

def main():
    summary = load_summary(SUMMARY_CSV)
    print('\nLoaded summary:')
    print(SUMMARY_CSV)
    print('\nRows used for plotting:')
    display_cols = ['endpoint', 'endpoint_display', 'method', 'method_display', 'n_analyzed', 'n_event', 'cindex', 'event_auroc']
    print(summary[display_cols].to_string(index=False))
    plot_visit02_style(summary=summary, metric='cindex', out_pdf=os.path.join(OUT_DIR, 'istroke_ascvd_visit02_style_cindex.pdf'), out_png=os.path.join(OUT_DIR, 'istroke_ascvd_visit02_style_cindex.png'), figsize=(0.8, 4.5))
    plot_visit02_style(summary=summary, metric='event_auroc', out_pdf=os.path.join(OUT_DIR, 'istroke_ascvd_visit02_style_auc.pdf'), out_png=os.path.join(OUT_DIR, 'istroke_ascvd_visit02_style_auc.png'), figsize=(6.8, 4.5))
    print('\nDone.')
    print('Output dir:', OUT_DIR)
if __name__ == '__main__':
    main()


### Figure 4c — Multimodal integration strategies, combined layout


In [ ]:
import os
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE = '.'
IN_DIR = os.path.join(BASE, 'code_3_3_save/compare_external_mean_fivefold_fullcox_visit02')
SUMMARY_CSV = os.path.join(IN_DIR, 'external_comparison_cindex_auc_summary.csv')
OUT_DIR = os.path.join('fig4_final', 'figures_visit02_style_istroke_ascvd_2x2_fixed')
os.makedirs(OUT_DIR, exist_ok=True)
COLOR_VISIT02 = '#8B1E1E'
COLOR_OTHER = '#A3A9A6'
TARGET_ENDPOINTS = ['istroke', 'ICVD']
ENDPOINT_DISPLAY = {'istroke': 'Ischemic stroke', 'ICVD': 'ASCVD'}
VISIT02_METHOD_RAW = 'Visit02 best multimodal'
METHOD_LABELS = {'External direct mean': 'Direct mean', 'Fivefold Cox weighted': '5-fold Cox weighted', 'Full-train Cox weighted': 'Full-train Cox weighted', 'Visit02 best multimodal': 'Latent-based multimodal aggregate'}
OTHER_METHOD_ORDER = ['External direct mean', 'Fivefold Cox weighted', 'Full-train Cox weighted']
METRICS = [('event_auroc', 'Event AUROC'), ('cindex', 'C-index')]
XLIM = (0.6, 0.75)
USE_RELATIVE_PERCENT_GAIN = True

def load_summary(summary_csv):
    if not os.path.exists(str(OUTPUT_ROOT / summary_csv if (OUTPUT_ROOT / summary_csv).exists() else INPUT_ROOT / summary_csv)):
        raise FileNotFoundError(f'Cannot find summary CSV:\n{summary_csv}\n\nRun the upstream analysis to generate external_comparison_cindex_auc_summary.csv.')
    df = pd.read_csv(str(OUTPUT_ROOT / summary_csv if (OUTPUT_ROOT / summary_csv).exists() else INPUT_ROOT / summary_csv)).rename(columns={'n': 'n_analyzed'})
    if 'event_auroc' not in df.columns and 'auc' in df.columns:
        df['event_auroc'] = df['auc']
    required = ['endpoint', 'method', 'n_analyzed', 'n_event', 'cindex', 'event_auroc']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'summary CSV missing columns: {missing}')
    df = df.copy()
    df = df[df['endpoint'].isin(TARGET_ENDPOINTS)].copy()
    if len(df) == 0:
        raise ValueError('No istroke / ICVD rows found in summary CSV. Please check endpoint names.')
    df['endpoint_display'] = df['endpoint'].map(ENDPOINT_DISPLAY)
    df['method_display'] = df['method'].map(METHOD_LABELS)
    df['method_display'] = df['method_display'].fillna(df['method'])
    df['event_auroc'] = pd.to_numeric(df['event_auroc'], errors='coerce')
    df['cindex'] = pd.to_numeric(df['cindex'], errors='coerce')
    df['n_analyzed'] = pd.to_numeric(df['n_analyzed'], errors='coerce')
    df['n_event'] = pd.to_numeric(df['n_event'], errors='coerce')
    return df

def prepare_endpoint_df(summary, endpoint):
    sub = summary[summary['endpoint'] == endpoint].copy()
    if len(sub) == 0:
        raise ValueError(f'No rows found for endpoint: {endpoint}')
    visit02_sub = sub[sub['method'] == VISIT02_METHOD_RAW].copy()
    if len(visit02_sub) == 0:
        raise ValueError(f"{endpoint}: cannot find method '{VISIT02_METHOD_RAW}' in summary CSV.")
    other = sub[sub['method'] != VISIT02_METHOD_RAW].copy()
    ordered_other_rows = []
    for m in OTHER_METHOD_ORDER:
        tmp = other[other['method'] == m].copy()
        if len(tmp) > 0:
            ordered_other_rows.append(tmp.iloc[[0]])
    remaining = other[~other['method'].isin(OTHER_METHOD_ORDER)].copy()
    if len(remaining) > 0:
        ordered_other_rows.append(remaining)
    if len(ordered_other_rows) > 0:
        other_ordered = pd.concat(ordered_other_rows, axis=0, ignore_index=True)
    else:
        other_ordered = pd.DataFrame(columns=sub.columns)
    plot_df = pd.concat([visit02_sub.iloc[[0]], other_ordered], axis=0, ignore_index=True)
    return plot_df

def draw_metric_panel(ax, plot_df, metric, show_ylabels=True, show_xlabel=True):
    plot_df = plot_df.copy()
    plot_df = plot_df.dropna(subset=[metric]).copy()
    y = np.arange(len(plot_df))
    values = plot_df[metric].astype(float).values
    labels = plot_df['method_display'].tolist()
    methods = plot_df['method'].tolist()
    colors = [COLOR_VISIT02 if m == VISIT02_METHOD_RAW else COLOR_OTHER for m in methods]
    visit02_vals = plot_df.loc[plot_df['method'] == VISIT02_METHOD_RAW, metric].values
    if len(visit02_vals) == 0:
        raise ValueError(f'Cannot find Visit02 value for metric: {metric}')
    visit02_val = float(visit02_vals[0])
    ax.barh(y, values, color=colors, height=0.85)
    ax.set_yticks(y)
    if show_ylabels:
        ax.set_yticklabels(labels, fontsize=10)
        ax.tick_params(axis='y', labelleft=True, left=True)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis='y', labelleft=False, left=True)
    ax.invert_yaxis()
    ax.axvline(visit02_val, linestyle='--', color=COLOR_VISIT02, linewidth=1.5, alpha=0.95, zorder=3)
    for (yy, val, method) in zip(y, values, methods):
        if np.isnan(val):
            continue
        if method == VISIT02_METHOD_RAW:
            txt_color = COLOR_VISIT02
            fs = 10.5
            fw = 'normal'
        else:
            txt_color = '#3A3A3A'
            fs = 9.6
            fw = 'normal'
        x_value_text = min(val + 0.003, XLIM[1] - 0.018)
        ax.text(x_value_text, yy, f'{val:.3f}', va='center', ha='left', fontsize=fs, color=txt_color, fontweight=fw, clip_on=False)
    gain_x = min(visit02_val + 0.016, XLIM[1] - 0.035)
    for (yy, val, method) in zip(y, values, methods):
        if method == VISIT02_METHOD_RAW:
            continue
        if np.isnan(val) or val == 0:
            continue
        if USE_RELATIVE_PERCENT_GAIN:
            gain = (visit02_val - val) / val * 100.0
            gain_text = f'{gain:+.1f}%'
        else:
            gain = (visit02_val - val) * 100.0
            gain_text = f'{gain:+.1f} pp'
        ax.text(gain_x, yy, gain_text, va='center', ha='left', fontsize=10.2, color=COLOR_VISIT02, clip_on=False)
    ax.set_xlim(*XLIM)
    ax.grid(axis='x', linestyle='--', alpha=0.25, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if show_xlabel:
        if metric == 'event_auroc':
            ax.set_xlabel('Event AUROC', fontsize=12)
        elif metric == 'cindex':
            ax.set_xlabel('C-index', fontsize=12)
    else:
        ax.set_xlabel('')
    return ax

def plot_2x2_auc_cindex(summary):
    (fig, axes) = plt.subplots(2, 2, figsize=(10.8, 6.4), sharex=True, sharey='row')
    for (row_idx, endpoint) in enumerate(TARGET_ENDPOINTS):
        plot_df = prepare_endpoint_df(summary, endpoint)
        n_val = int(plot_df.loc[plot_df['method'] == VISIT02_METHOD_RAW, 'n_analyzed'].iloc[0])
        e_val = int(plot_df.loc[plot_df['method'] == VISIT02_METHOD_RAW, 'n_event'].iloc[0])
        for (col_idx, (metric, metric_label)) in enumerate(METRICS):
            ax = axes[row_idx, col_idx]
            draw_metric_panel(ax=ax, plot_df=plot_df, metric=metric, show_ylabels=col_idx == 0, show_xlabel=row_idx == 1)
            if row_idx == 0:
                ax.set_title(metric_label, fontsize=15, pad=10)
        row_title = f'{ENDPOINT_DISPLAY.get(endpoint, endpoint)} (analyzed n={n_val}, events={e_val})'
        if row_idx == 0:
            y_pos = 0.965
        else:
            y_pos = 0.505
        fig.text(0.52, y_pos, row_title, ha='center', va='center', fontsize=18, fontweight='bold')
    for ax in axes.flat:
        ax.set_xticks([0.6, 0.65, 0.7, 0.75])
        ax.tick_params(axis='x', labelsize=11)
    for ax in axes[0, :]:
        ax.tick_params(labelbottom=False)
    plt.subplots_adjust(left=0.34, right=0.97, top=0.88, bottom=0.13, wspace=0.08, hspace=0.43)
    out_pdf = os.path.join(OUT_DIR, 'istroke_ascvd_auc_cindex_2x2_fixed.pdf')
    out_png = os.path.join(OUT_DIR, 'istroke_ascvd_auc_cindex_2x2_fixed.png')
    plt.savefig(out_pdf, transparent=True, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, transparent=True, bbox_inches='tight')
    print('Saved:', out_pdf)
    print('Saved:', out_png)
    plt.show()

def main():
    summary = load_summary(SUMMARY_CSV)
    print('\nLoaded summary:')
    print(SUMMARY_CSV)
    display_cols = ['endpoint', 'endpoint_display', 'method', 'method_display', 'n_analyzed', 'n_event', 'cindex', 'event_auroc']
    print('\nRows used for plotting:')
    print(summary[display_cols].to_string(index=False))
    print('\nGain definition:')
    if USE_RELATIVE_PERCENT_GAIN:
        print('gain = (latent - other) / other * 100%')
    else:
        print('gain = (latent - other) * 100 percentage points')
    plot_2x2_auc_cindex(summary)
    print('\nDone.')
    print('Output dir:', OUT_DIR)
if __name__ == '__main__':
    main()


### Figure 4d — Event enrichment and capture, discrete groups


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE = '.'
IN_DIR = os.path.join(BASE, 'code_3_3_save/compare_external_mean_fivefold_fullcox_visit02')
PRED_DIR = os.path.join(IN_DIR, 'preds')
OUT_DIR = os.path.join('fig4_final', 'ICVD_top_risk_capture_rate_combined_update')
os.makedirs(OUT_DIR, exist_ok=True)
ENDPOINT = 'ICVD'
ENDPOINT_DISPLAY = 'ASCVD'
METHOD_FILES = {'Direct mean': 'external_direct_mean', 'latent-based multimodel aggregate': 'visit02_best_multimodal'}
METHOD_ORDER = ['Direct mean', 'latent-based multimodel aggregate']
COLOR_DIRECT = '#A3A9A6'
COLOR_LATENT = '#8B1E1E'
METHOD_COLORS = {'Direct mean': COLOR_DIRECT, 'latent-based multimodel aggregate': COLOR_LATENT}
TOP_QS = [0.05, 0.1, 0.2, 0.3]

def rank_percentile_high_is_high(x):
    s = pd.Series(x).astype(float)
    if len(s) <= 1:
        return np.zeros(len(s), dtype=float)
    r = s.rank(method='average').values
    return (r - 1) / (len(r) - 1 + 1e-08)

def get_pred_file(endpoint, method_key):
    return os.path.join(PRED_DIR, f'{endpoint}_{method_key}_external_aligned_pred.csv')

def load_one_method(endpoint, method_label, method_key):
    path = get_pred_file(endpoint, method_key)
    if not os.path.exists(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path)):
        raise FileNotFoundError(f'Cannot find prediction file:\n{path}\n\nRun the upstream analysis and supply the matching prediction files.')
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    if 'ID' in df.columns:
        df = df.rename(columns={'ID': 'pid'})
    required = ['pid', 'time', 'event', 'risk_norm']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{path} missing columns: {missing}')
    df = df[required].copy()
    df['pid'] = pd.to_numeric(df['pid'], errors='coerce')
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df['event'] = pd.to_numeric(df['event'], errors='coerce').fillna(0).astype(int)
    df['risk_norm'] = pd.to_numeric(df['risk_norm'], errors='coerce')
    df = df.dropna(subset=['pid', 'time', 'event', 'risk_norm']).copy()
    df['pid'] = df['pid'].astype(int)
    df = df.rename(columns={'risk_norm': f'{method_label}_risk'})
    return (df, path)

def load_aligned_icvd():
    loaded = []
    file_rows = []
    for method_label in METHOD_ORDER:
        method_key = METHOD_FILES[method_label]
        (df, path) = load_one_method(endpoint=ENDPOINT, method_label=method_label, method_key=method_key)
        loaded.append(df)
        file_rows.append({'endpoint': ENDPOINT, 'method': method_label, 'file': path, 'n': len(df), 'n_event': int(df['event'].sum())})
    merged = loaded[0]
    for df in loaded[1:]:
        merged = merged.merge(df, on=['pid', 'time', 'event'], how='inner')
    if len(merged) == 0:
        raise ValueError('Zero shared pid between Direct mean and latent-based prediction.')
    for method_label in METHOD_ORDER:
        risk_col = f'{method_label}_risk'
        pct_col = f'{method_label}_risk_percentile'
        merged[pct_col] = rank_percentile_high_is_high(merged[risk_col].values)
    merged['is_event'] = merged['event'].astype(int)
    return (merged, pd.DataFrame(file_rows))

def summarize_top_risk(df):
    rows = []
    n_total = len(df)
    n_event_total = int(df['is_event'].sum())
    event_df_all = df[df['is_event'] == 1].copy()
    overall_event_rate = n_event_total / n_total if n_total > 0 else np.nan
    overall_mean_event_time = event_df_all['time'].mean() if len(event_df_all) > 0 else np.nan
    overall_median_event_time = event_df_all['time'].median() if len(event_df_all) > 0 else np.nan
    for method_label in METHOD_ORDER:
        pct_col = f'{method_label}_risk_percentile'
        for q in TOP_QS:
            cutoff = float(df[pct_col].quantile(1 - q))
            top = df[df[pct_col] >= cutoff].copy()
            top_event = top[top['is_event'] == 1].copy()
            top_n = len(top)
            event_in_top = int(top['is_event'].sum())
            event_capture = event_in_top / n_event_total if n_event_total > 0 else np.nan
            event_rate_in_top = event_in_top / top_n if top_n > 0 else np.nan
            event_enrichment = event_rate_in_top / overall_event_rate if overall_event_rate > 0 else np.nan
            mean_event_time_in_top = top_event['time'].mean() if len(top_event) > 0 else np.nan
            median_event_time_in_top = top_event['time'].median() if len(top_event) > 0 else np.nan
            rows.append({'endpoint': ENDPOINT, 'endpoint_display': ENDPOINT_DISPLAY, 'method': method_label, 'top_q': q, 'top_percent': int(round(q * 100)), 'n_total': n_total, 'n_event_total': n_event_total, 'top_n': top_n, 'event_in_top': event_in_top, 'event_capture': event_capture, 'event_rate_in_top': event_rate_in_top, 'event_enrichment': event_enrichment, 'mean_event_time_in_top': mean_event_time_in_top, 'median_event_time_in_top': median_event_time_in_top, 'overall_event_rate': overall_event_rate, 'overall_mean_event_time': overall_mean_event_time, 'overall_median_event_time': overall_median_event_time, 'risk_percentile_cutoff': cutoff})
    return pd.DataFrame(rows)

def save_separate_legend(out_pdf, out_png):
    (fig, ax) = plt.subplots(figsize=(4.8, 0.6))
    handles = []
    for method_label in METHOD_ORDER:
        h = ax.bar(0, 0, color=METHOD_COLORS[method_label], label=method_label)
        handles.append(h[0])
    ax.legend(handles=handles, labels=METHOD_ORDER, frameon=False, ncol=2, fontsize=9, loc='center')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(out_pdf, transparent=True, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, transparent=True, bbox_inches='tight')
    plt.close()
    print('Saved legend:', out_pdf)
    print('Saved legend:', out_png)

def plot_combined_event_capture_and_rate(summary_df, out_pdf, out_png, show_legend=False):
    qs = sorted(summary_df['top_percent'].unique().tolist())
    (fig, axes) = plt.subplots(2, 1, figsize=(5.8, 5.2), sharex=True, gridspec_kw={'height_ratios': [1, 1], 'hspace': 0.08})
    ax_capture = axes[0]
    ax_rate = axes[1]
    x = np.arange(len(qs))
    width = 0.34
    panel_settings = [(ax_capture, 'event_capture', 'Event capture', 'A. Event capture in predicted top-risk group'), (ax_rate, 'event_rate_in_top', 'Observed event rate', 'B. Observed event rate in predicted top-risk group')]
    for (ax, metric, ylabel, title) in panel_settings:
        max_y = 0
        for (i, method_label) in enumerate(METHOD_ORDER):
            tmp = summary_df[summary_df['method'] == method_label].copy()
            vals = []
            event_counts = []
            mean_times = []
            for q in qs:
                one = tmp[tmp['top_percent'] == q]
                if len(one) == 0:
                    vals.append(np.nan)
                    event_counts.append(0)
                    mean_times.append(np.nan)
                else:
                    vals.append(float(one[metric].iloc[0]))
                    event_counts.append(int(one['event_in_top'].iloc[0]))
                    mean_times.append(float(one['mean_event_time_in_top'].iloc[0]))
            vals = np.array(vals, dtype=float)
            event_counts = np.array(event_counts, dtype=int)
            mean_times = np.array(mean_times, dtype=float)
            if np.isfinite(vals).any():
                max_y = max(max_y, np.nanmax(vals))
            offset = -width / 2 if i == 0 else width / 2
            bars = ax.bar(x + offset, vals, width=width, color=METHOD_COLORS[method_label], label=method_label)
            for (bar, val) in zip(bars, vals):
                if np.isnan(val):
                    continue
                label = f'{val * 100:.1f}%'
                ax.text(bar.get_x() + bar.get_width() / 2, val + 0.012, label, ha='center', va='bottom', fontsize=8.5, rotation=0)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=11, loc='left', pad=6)
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ymax = min(1.05, max(0.15, max_y + 0.14))
        ax.set_ylim(0, ymax)
    overall_event_rate = float(summary_df['overall_event_rate'].dropna().iloc[0])
    ax_rate.axhline(overall_event_rate, linestyle='--', color='black', linewidth=1.0, alpha=0.55)
    ax_rate.text(0.99, overall_event_rate + 0.006, f'Overall event rate = {overall_event_rate * 100:.1f}%', transform=ax_rate.get_yaxis_transform(), ha='right', va='bottom', fontsize=8, color='black')
    ax_rate.set_xticks(x)
    ax_rate.set_xticklabels([f'Top {q}%' for q in qs], fontsize=10)
    ax_rate.set_xlabel('Predicted high-risk group', fontsize=10)
    if show_legend:
        ax_capture.legend(frameon=False, fontsize=8, loc='upper left')
    else:
        leg = ax_capture.get_legend()
        if leg is not None:
            leg.remove()
    fig.suptitle(f'{ENDPOINT_DISPLAY}: event capture and event rate in top-risk groups', fontsize=12, y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(out_pdf, transparent=True, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, transparent=True, bbox_inches='tight')
    plt.close()
    print('Saved combined figure:', out_pdf)
    print('Saved combined figure:', out_png)

def print_top20_average_time(summary_df):
    top20 = summary_df[summary_df['top_percent'] == 20].copy()
    print('\n================================================')
    print('TOP 20% AVERAGE EVENT TIME')
    print('================================================')
    for method_label in METHOD_ORDER:
        one = top20[top20['method'] == method_label]
        if len(one) == 0:
            print(f'{method_label}: NA')
            continue
        row = one.iloc[0]
        mean_t = row['mean_event_time_in_top']
        median_t = row['median_event_time_in_top']
        event_n = int(row['event_in_top'])
        top_n = int(row['top_n'])
        event_rate = row['event_rate_in_top']
        event_capture = row['event_capture']
        print(f'\n{method_label}')
        print(f'  top_n                    : {top_n}')
        print(f'  event_in_top             : {event_n}')
        print(f'  event_capture            : {event_capture * 100:.2f}%')
        print(f'  event_rate_in_top        : {event_rate * 100:.2f}%')
        if np.isnan(mean_t):
            print('  mean_event_time_in_top20 : NA')
        else:
            print(f'  mean_event_time_in_top20 : {mean_t:.4f}')
        if np.isnan(median_t):
            print('  median_event_time_top20  : NA')
        else:
            print(f'  median_event_time_top20  : {median_t:.4f}')

def main():
    print('\n================================================')
    print(f'Endpoint: {ENDPOINT} ({ENDPOINT_DISPLAY})')
    print('================================================')
    (df, file_rows) = load_aligned_icvd()
    print('\nInput files:')
    print(file_rows.to_string(index=False))
    print('\nAligned data:')
    print(f'  n          : {len(df)}')
    print(f"  n_event    : {int(df['is_event'].sum())}")
    print(f"  event rate : {df['is_event'].mean():.4f}")
    event_df = df[df['is_event'] == 1].copy()
    print('\nAll ICVD event samples time summary:')
    print(f"  mean event time   : {event_df['time'].mean():.4f}")
    print(f"  median event time : {event_df['time'].median():.4f}")
    print(f"  min event time    : {event_df['time'].min():.4f}")
    print(f"  max event time    : {event_df['time'].max():.4f}")
    aligned_out = os.path.join(OUT_DIR, 'ICVD_direct_vs_latent_aligned_predictions.csv')
    df.to_csv(aligned_out, index=False)
    print('\nSaved aligned predictions:')
    print(aligned_out)
    summary_df = summarize_top_risk(df)
    summary_out = os.path.join(OUT_DIR, 'ICVD_top_risk_event_capture_rate_summary.csv')
    summary_df.to_csv(summary_out, index=False)
    print('\nSaved top-risk summary:')
    print(summary_out)
    display_cols = ['method', 'top_percent', 'top_n', 'event_in_top', 'event_capture', 'event_rate_in_top', 'event_enrichment', 'mean_event_time_in_top', 'median_event_time_in_top']
    print('\nTop-risk summary:')
    print(summary_df[display_cols].to_string(index=False))
    print_top20_average_time(summary_df)
    plot_combined_event_capture_and_rate(summary_df=summary_df, out_pdf=os.path.join(OUT_DIR, 'ICVD_top_risk_event_capture_and_rate_combined.pdf'), out_png=os.path.join(OUT_DIR, 'ICVD_top_risk_event_capture_and_rate_combined.png'), show_legend=False)
    save_separate_legend(out_pdf=os.path.join(OUT_DIR, 'ICVD_top_risk_event_capture_and_rate_legend.pdf'), out_png=os.path.join(OUT_DIR, 'ICVD_top_risk_event_capture_and_rate_legend.png'))
    print('\nDone.')
    print('Output dir:', OUT_DIR)
if __name__ == '__main__':
    main()


### Figure 4d — Event enrichment and capture curves


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE = '.'
IN_DIR = os.path.join(BASE, 'code_3_3_save/compare_external_mean_fivefold_fullcox_visit02')
PRED_DIR = os.path.join(IN_DIR, 'preds')
OUT_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_top_risk_capture_rate_combined_update')
os.makedirs(OUT_DIR, exist_ok=True)
ENDPOINT = 'ICVD'
ENDPOINT_DISPLAY = 'ASCVD'
METHOD_FILES = {'Direct mean': 'external_direct_mean', 'latent-based multimodel aggregate': 'visit02_best_multimodal'}
METHOD_ORDER = ['Direct mean', 'latent-based multimodel aggregate']
METHOD_COLORS = {'Direct mean': '#A3A9A6', 'latent-based multimodel aggregate': '#8B1E1E'}
METHOD_LABELS = {'Direct mean': 'Direct mean', 'latent-based multimodel aggregate': 'Multimodal latent'}
TOP_PERCENTS = np.arange(1, 41, 1)
TOP_QS = TOP_PERCENTS / 100.0
FIG_W_MM = 90
FIG_H_MM = 45
MM_TO_INCH = 1 / 25.4
FIGSIZE = (FIG_W_MM * MM_TO_INCH, FIG_H_MM * MM_TO_INCH)

def rank_percentile_high_is_high(x):
    s = pd.Series(x).astype(float)
    if len(s) <= 1:
        return np.zeros(len(s), dtype=float)
    r = s.rank(method='average').values
    return (r - 1) / (len(r) - 1 + 1e-08)

def get_pred_file(endpoint, method_key):
    return os.path.join(PRED_DIR, f'{endpoint}_{method_key}_external_aligned_pred.csv')

def load_one_method(endpoint, method_label, method_key):
    path = get_pred_file(endpoint, method_key)
    if not os.path.exists(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path)):
        raise FileNotFoundError(f'Cannot find prediction file:\n{path}\n\nGenerate the required prediction files before running this analysis.')
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    if 'ID' in df.columns:
        df = df.rename(columns={'ID': 'pid'})
    required = ['pid', 'time', 'event', 'risk_norm']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{path} missing columns: {missing}')
    df = df[required].copy()
    df['pid'] = pd.to_numeric(df['pid'], errors='coerce')
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df['event'] = pd.to_numeric(df['event'], errors='coerce').fillna(0).astype(int)
    df['risk_norm'] = pd.to_numeric(df['risk_norm'], errors='coerce')
    df = df.dropna(subset=['pid', 'time', 'event', 'risk_norm']).copy()
    df['pid'] = df['pid'].astype(int)
    df = df.rename(columns={'risk_norm': f'{method_label}_risk'})
    return (df, path)

def load_aligned_icvd():
    loaded = []
    file_rows = []
    for method_label in METHOD_ORDER:
        method_key = METHOD_FILES[method_label]
        (df, path) = load_one_method(endpoint=ENDPOINT, method_label=method_label, method_key=method_key)
        loaded.append(df)
        file_rows.append({'endpoint': ENDPOINT, 'method': method_label, 'file': path, 'n': len(df), 'n_event': int(df['event'].sum())})
    merged = loaded[0]
    for df in loaded[1:]:
        merged = merged.merge(df, on=['pid', 'time', 'event'], how='inner')
    if len(merged) == 0:
        raise ValueError('Zero shared pid between Direct mean and latent-based prediction.')
    for method_label in METHOD_ORDER:
        risk_col = f'{method_label}_risk'
        pct_col = f'{method_label}_risk_percentile'
        merged[pct_col] = rank_percentile_high_is_high(merged[risk_col].values)
    merged['is_event'] = merged['event'].astype(int)
    return (merged, pd.DataFrame(file_rows))

def summarize_top_risk_curves(df):
    rows = []
    n_total = len(df)
    n_event_total = int(df['is_event'].sum())
    overall_event_rate = n_event_total / n_total if n_total > 0 else np.nan
    for method_label in METHOD_ORDER:
        pct_col = f'{method_label}_risk_percentile'
        for (top_percent, q) in zip(TOP_PERCENTS, TOP_QS):
            cutoff = float(df[pct_col].quantile(1 - q))
            top = df[df[pct_col] >= cutoff].copy()
            top_n = len(top)
            event_in_top = int(top['is_event'].sum())
            observed_event_rate = event_in_top / top_n if top_n > 0 else np.nan
            event_capture = event_in_top / n_event_total if n_event_total > 0 else np.nan
            rows.append({'endpoint': ENDPOINT, 'endpoint_display': ENDPOINT_DISPLAY, 'method': method_label, 'top_percent': int(top_percent), 'top_q': float(q), 'n_total': n_total, 'n_event_total': n_event_total, 'top_n': top_n, 'event_in_top': event_in_top, 'observed_event_rate': observed_event_rate, 'event_capture': event_capture, 'overall_event_rate': overall_event_rate, 'risk_percentile_cutoff': cutoff})
    return pd.DataFrame(rows)

def top_percent_to_x(top_percent):
    top_percent = np.asarray(top_percent, dtype=float)
    return np.log1p(top_percent)

def setup_loglike_xaxis(ax):
    xticks_raw = [0, 2, 5, 10, 20, 30, 40]
    xticks_pos = top_percent_to_x(xticks_raw)
    ax.set_xlim(top_percent_to_x(0), top_percent_to_x(40))
    ax.set_xticks(xticks_pos)
    ax.set_xticklabels([str(x) for x in xticks_raw], fontsize=6.5)
    ax.set_xlabel('Predicted top-risk group (%)', fontsize=7, labelpad=2)

def plot_icvd_rate_and_capture_top40(summary_df, out_pdf, out_png):
    (fig, axes) = plt.subplots(1, 2, figsize=FIGSIZE, sharex=True, constrained_layout=True)
    ax_rate = axes[0]
    ax_capture = axes[1]
    top5_x = top_percent_to_x(5)
    panel_settings = [(ax_rate, 'observed_event_rate', 'Observed event rate\nin predicted top-risk group', 'Observed event rate'), (ax_capture, 'event_capture', 'B. Event capture\nin predicted top-risk group', 'Event capture')]
    for (ax, metric, title, ylabel) in panel_settings:
        ymax = 0
        for method_label in METHOD_ORDER:
            tmp = summary_df[summary_df['method'] == method_label].sort_values('top_percent').copy()
            x = top_percent_to_x(tmp['top_percent'].values)
            y = tmp[metric].values.astype(float)
            if np.isfinite(y).any():
                ymax = max(ymax, np.nanmax(y))
            ax.plot(x, y, linewidth=1.6, marker='o', markersize=2.2, color=METHOD_COLORS[method_label], label=METHOD_LABELS[method_label], zorder=2)
            one5 = tmp[tmp['top_percent'] == 5]
            if len(one5) > 0:
                y5 = float(one5[metric].iloc[0])
                ax.scatter([top5_x], [y5], s=24, color=METHOD_COLORS[method_label], edgecolor='white', linewidth=0.6, zorder=5)
        ax.axvline(top5_x, linestyle='--', color='black', linewidth=0.8, alpha=0.55, zorder=1)
        ax.text(top5_x, 0.98, 'Top 5%', transform=ax.get_xaxis_transform(), ha='center', va='top', fontsize=5.8, color='black')
        ax.set_title(title, fontsize=7.5, loc='left', pad=2)
        ax.set_ylabel(ylabel, fontsize=7, labelpad=2)
        ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.35)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='both', labelsize=6.5, length=2, width=0.6, pad=1.5)
        ax.set_ylim(0, min(1.02, max(0.12, ymax * 1.16)))
        setup_loglike_xaxis(ax)
    overall_event_rate = float(summary_df['overall_event_rate'].dropna().iloc[0])
    ax_rate.axhline(overall_event_rate, linestyle='--', color='black', linewidth=0.8, alpha=0.55)
    ax_rate.text(top_percent_to_x(39.5), overall_event_rate, f'Overall {overall_event_rate * 100:.1f}%', ha='right', va='bottom', fontsize=5.8, color='black')
    ax_rate.legend(frameon=False, fontsize=5.8, loc='upper right', handlelength=1.4, borderaxespad=0.2)
    fig.savefig(out_pdf, transparent=True)
    fig.savefig(out_png, dpi=600, transparent=True)
    plt.close(fig)
    print('Saved figure:')
    print(out_pdf)
    print(out_png)

def main():
    print('\n================================================')
    print(f'Endpoint: {ENDPOINT} ({ENDPOINT_DISPLAY})')
    print('================================================')
    (df, file_rows) = load_aligned_icvd()
    print('\nInput files:')
    print(file_rows.to_string(index=False))
    print('\nAligned data:')
    print(f'  n          : {len(df)}')
    print(f"  n_event    : {int(df['is_event'].sum())}")
    print(f"  event rate : {df['is_event'].mean():.4f}")
    aligned_out = os.path.join(OUT_DIR, 'ICVD_direct_vs_latent_aligned_predictions_for_top40_curves.csv')
    df.to_csv(aligned_out, index=False)
    summary_df = summarize_top_risk_curves(df)
    summary_out = os.path.join(OUT_DIR, 'ICVD_observed_event_rate_and_capture_curves_top40_summary.csv')
    summary_df.to_csv(summary_out, index=False)
    print('\nSaved summary:')
    print(summary_out)
    display_cols = ['method', 'top_percent', 'top_n', 'event_in_top', 'observed_event_rate', 'event_capture']
    print('\nTop-risk curve summary:')
    print(summary_df[display_cols].to_string(index=False))
    out_pdf = os.path.join(OUT_DIR, 'ICVD_observed_event_rate_and_capture_curves_top40_combined.pdf')
    out_png = os.path.join(OUT_DIR, 'ICVD_observed_event_rate_and_capture_curves_top40_combined.png')
    plot_icvd_rate_and_capture_top40(summary_df=summary_df, out_pdf=out_pdf, out_png=out_png)
    print('\nDone.')
    print('Output dir:', OUT_DIR)
if __name__ == '__main__':
    main()


### Figure 4d — Risk-segment significance analysis


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, chi2_contingency
from scipy.stats import chi2
from statsmodels.stats.proportion import proportions_ztest
warnings.filterwarnings('ignore')
BASE = '.'
IN_DIR = os.path.join(BASE, 'code_3_3_save/compare_external_mean_fivefold_fullcox_visit02_aligned')
PRED_DIR = os.path.join(IN_DIR, 'preds')
OUT_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_top_risk_segment_fast_significance')
os.makedirs(OUT_DIR, exist_ok=True)
ENDPOINT = 'ICVD'
ENDPOINT_DISPLAY = 'ASCVD'
METHOD_FILES = {'Direct mean': 'external_direct_mean', 'latent-based multimodel aggregate': 'visit02_best_multimodal'}
METHOD_ORDER = ['Direct mean', 'latent-based multimodel aggregate']
DIRECT_METHOD = 'Direct mean'
LATENT_METHOD = 'latent-based multimodel aggregate'
SEGMENTS = [{'segment': 'top0_to_top5', 'display': 'Top 0-5%', 'low_pct': 0.95, 'high_pct': 1.000001, 'interpretation': 'extreme top-risk group'}, {'segment': 'top5_to_top40', 'display': 'Top 5-40%', 'low_pct': 0.6, 'high_pct': 0.95, 'interpretation': 'intermediate high-risk group excluding top5%'}, {'segment': 'top5_to_top100', 'display': 'Top 5-100%', 'low_pct': -1e-06, 'high_pct': 0.95, 'interpretation': 'remaining population excluding top5%'}]

def rank_percentile_high_is_high(x):
    s = pd.Series(x).astype(float)
    if len(s) <= 1:
        return np.zeros(len(s), dtype=float)
    r = s.rank(method='average').values
    return (r - 1) / (len(r) - 1 + 1e-08)

def get_pred_file(endpoint, method_key):
    return os.path.join(PRED_DIR, f'{endpoint}_{method_key}_external_aligned_pred.csv')

def load_one_method(endpoint, method_label, method_key):
    path = get_pred_file(endpoint, method_key)
    if not os.path.exists(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path)):
        raise FileNotFoundError(f'Cannot find prediction file:\n{path}')
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    if 'ID' in df.columns:
        df = df.rename(columns={'ID': 'pid'})
    required = ['pid', 'time', 'event', 'risk_norm']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{path} missing columns: {missing}')
    df = df[required].copy()
    df['pid'] = pd.to_numeric(df['pid'], errors='coerce')
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df['event'] = pd.to_numeric(df['event'], errors='coerce').fillna(0).astype(int)
    df['risk_norm'] = pd.to_numeric(df['risk_norm'], errors='coerce')
    df = df.dropna(subset=['pid', 'time', 'event', 'risk_norm']).copy()
    df['pid'] = df['pid'].astype(int)
    df = df.rename(columns={'risk_norm': f'{method_label}_risk'})
    return (df, path)

def load_aligned_icvd():
    loaded = []
    file_rows = []
    for method_label in METHOD_ORDER:
        method_key = METHOD_FILES[method_label]
        (df, path) = load_one_method(endpoint=ENDPOINT, method_label=method_label, method_key=method_key)
        loaded.append(df)
        file_rows.append({'endpoint': ENDPOINT, 'method': method_label, 'file': path, 'n': len(df), 'n_event': int(df['event'].sum())})
    merged = loaded[0]
    for df in loaded[1:]:
        merged = merged.merge(df, on=['pid', 'time', 'event'], how='inner')
    if len(merged) == 0:
        raise ValueError('Zero shared pid between Direct mean and latent-based prediction.')
    merged['is_event'] = merged['event'].astype(int)
    for method_label in METHOD_ORDER:
        risk_col = f'{method_label}_risk'
        pct_col = f'{method_label}_risk_percentile'
        merged[pct_col] = rank_percentile_high_is_high(merged[risk_col].values)
    return (merged, pd.DataFrame(file_rows))

def get_segment_mask(df, method_label, low_pct, high_pct):
    pct_col = f'{method_label}_risk_percentile'
    mask = (df[pct_col] >= low_pct) & (df[pct_col] < high_pct)
    return mask

def summarize_mask(df, mask):
    sub = df.loc[mask].copy()
    n = len(sub)
    event_n = int(sub['is_event'].sum())
    non_event_n = int(n - event_n)
    event_rate = event_n / n if n > 0 else np.nan
    total_events = int(df['is_event'].sum())
    event_capture = event_n / total_events if total_events > 0 else np.nan
    return {'n': n, 'event_n': event_n, 'non_event_n': non_event_n, 'event_rate': event_rate, 'event_capture': event_capture}

def mcnemar_exact_binomial(b, c, alternative='two-sided'):
    n = b + c
    if n == 0:
        return {'mcnemar_b_latent_only': b, 'mcnemar_c_direct_only': c, 'mcnemar_n_discordant': n, 'p_mcnemar_exact': np.nan}
    try:
        from scipy.stats import binomtest
        if alternative == 'greater':
            p = binomtest(k=b, n=n, p=0.5, alternative='greater').pvalue
        elif alternative == 'less':
            p = binomtest(k=b, n=n, p=0.5, alternative='less').pvalue
        else:
            p = binomtest(k=b, n=n, p=0.5, alternative='two-sided').pvalue
    except Exception:
        from scipy.stats import binom_test
        if alternative == 'greater':
            p = binom_test(x=b, n=n, p=0.5, alternative='greater')
        elif alternative == 'less':
            p = binom_test(x=b, n=n, p=0.5, alternative='less')
        else:
            p = binom_test(x=b, n=n, p=0.5, alternative='two-sided')
    return {'mcnemar_b_latent_only': b, 'mcnemar_c_direct_only': c, 'mcnemar_n_discordant': n, 'p_mcnemar_exact': p}

def test_one_segment(df, seg):
    low_pct = seg['low_pct']
    high_pct = seg['high_pct']
    latent_mask = get_segment_mask(df=df, method_label=LATENT_METHOD, low_pct=low_pct, high_pct=high_pct)
    direct_mask = get_segment_mask(df=df, method_label=DIRECT_METHOD, low_pct=low_pct, high_pct=high_pct)
    latent_sum = summarize_mask(df=df, mask=latent_mask)
    direct_sum = summarize_mask(df=df, mask=direct_mask)
    table_event_rate = np.array([[latent_sum['event_n'], latent_sum['non_event_n']], [direct_sum['event_n'], direct_sum['non_event_n']]])
    try:
        (odds_ratio_fisher, p_fisher_two_sided) = fisher_exact(table_event_rate, alternative='two-sided')
        (odds_ratio_fisher_greater, p_fisher_latent_gt_direct) = fisher_exact(table_event_rate, alternative='greater')
    except Exception:
        odds_ratio_fisher = np.nan
        p_fisher_two_sided = np.nan
        odds_ratio_fisher_greater = np.nan
        p_fisher_latent_gt_direct = np.nan
    try:
        (z_stat, p_prop_two_sided) = proportions_ztest(count=np.array([latent_sum['event_n'], direct_sum['event_n']]), nobs=np.array([latent_sum['n'], direct_sum['n']]), alternative='two-sided')
        (z_stat_greater, p_prop_latent_gt_direct) = proportions_ztest(count=np.array([latent_sum['event_n'], direct_sum['event_n']]), nobs=np.array([latent_sum['n'], direct_sum['n']]), alternative='larger')
    except Exception:
        z_stat = np.nan
        p_prop_two_sided = np.nan
        z_stat_greater = np.nan
        p_prop_latent_gt_direct = np.nan
    try:
        (chi2_stat, p_chi2, dof, expected) = chi2_contingency(table_event_rate)
    except Exception:
        chi2_stat = np.nan
        p_chi2 = np.nan
    event_df = df[df['is_event'] == 1].copy()
    latent_event_mask = latent_mask.loc[event_df.index]
    direct_event_mask = direct_mask.loc[event_df.index]
    b_latent_only = int(((latent_event_mask == True) & (direct_event_mask == False)).sum())
    c_direct_only = int(((latent_event_mask == False) & (direct_event_mask == True)).sum())
    mcnemar_two = mcnemar_exact_binomial(b=b_latent_only, c=c_direct_only, alternative='two-sided')
    mcnemar_greater = mcnemar_exact_binomial(b=b_latent_only, c=c_direct_only, alternative='greater')
    out = {'endpoint': ENDPOINT, 'endpoint_display': ENDPOINT_DISPLAY, 'segment': seg['segment'], 'segment_display': seg['display'], 'segment_interpretation': seg['interpretation'], 'low_pct': low_pct, 'high_pct': high_pct, 'n_total': len(df), 'n_event_total': int(df['is_event'].sum()), 'latent_n': latent_sum['n'], 'latent_event_n': latent_sum['event_n'], 'latent_non_event_n': latent_sum['non_event_n'], 'latent_event_rate': latent_sum['event_rate'], 'latent_event_capture': latent_sum['event_capture'], 'direct_n': direct_sum['n'], 'direct_event_n': direct_sum['event_n'], 'direct_non_event_n': direct_sum['non_event_n'], 'direct_event_rate': direct_sum['event_rate'], 'direct_event_capture': direct_sum['event_capture'], 'event_n_difference_latent_minus_direct': latent_sum['event_n'] - direct_sum['event_n'], 'event_rate_difference_latent_minus_direct': latent_sum['event_rate'] - direct_sum['event_rate'], 'event_capture_difference_latent_minus_direct': latent_sum['event_capture'] - direct_sum['event_capture'], 'event_rate_test_table': table_event_rate.tolist(), 'odds_ratio_fisher_event_rate': odds_ratio_fisher, 'p_fisher_event_rate_two_sided': p_fisher_two_sided, 'p_fisher_event_rate_latent_gt_direct': p_fisher_latent_gt_direct, 'z_stat_proportion_event_rate': z_stat, 'p_proportion_event_rate_two_sided': p_prop_two_sided, 'p_proportion_event_rate_latent_gt_direct': p_prop_latent_gt_direct, 'chi2_event_rate': chi2_stat, 'p_chi2_event_rate': p_chi2, 'mcnemar_b_event_latent_only': b_latent_only, 'mcnemar_c_event_direct_only': c_direct_only, 'mcnemar_discordant_event_n': b_latent_only + c_direct_only, 'p_mcnemar_event_capture_two_sided': mcnemar_two['p_mcnemar_exact'], 'p_mcnemar_event_capture_latent_gt_direct': mcnemar_greater['p_mcnemar_exact']}
    return out

def main():
    print('\n================================================')
    print(f'Endpoint: {ENDPOINT} ({ENDPOINT_DISPLAY})')
    print('Fast segment-level significance test')
    print('================================================')
    (df, file_rows) = load_aligned_icvd()
    print('\nInput files:')
    print(file_rows.to_string(index=False))
    print('\nAligned data:')
    print(f'  n          : {len(df)}')
    print(f"  n_event    : {int(df['is_event'].sum())}")
    print(f"  event rate : {df['is_event'].mean():.4f}")
    aligned_out = os.path.join(OUT_DIR, 'ICVD_direct_vs_latent_aligned_predictions_with_risk_percentiles.csv')
    df.to_csv(aligned_out, index=False)
    print('\nSaved aligned data:')
    print(aligned_out)
    rows = []
    for seg in SEGMENTS:
        print('\nTesting segment:', seg['display'])
        out = test_one_segment(df=df, seg=seg)
        rows.append(out)
        print(f"  latent event rate: {out['latent_event_rate']:.4f} ({out['latent_event_n']}/{out['latent_n']})")
        print(f"  direct event rate: {out['direct_event_rate']:.4f} ({out['direct_event_n']}/{out['direct_n']})")
        print(f"  Fisher one-sided latent > direct: {out['p_fisher_event_rate_latent_gt_direct']:.4g}")
        print(f"  Event capture McNemar one-sided latent > direct: {out['p_mcnemar_event_capture_latent_gt_direct']:.4g}")
    res = pd.DataFrame(rows)
    out_csv = os.path.join(OUT_DIR, 'ICVD_fast_segment_event_rate_and_capture_significance.csv')
    res.to_csv(out_csv, index=False)
    print('\n================================================')
    print('Fast segment-level results')
    print('================================================')
    show_cols = ['segment_display', 'latent_n', 'latent_event_n', 'latent_event_rate', 'latent_event_capture', 'direct_n', 'direct_event_n', 'direct_event_rate', 'direct_event_capture', 'event_rate_difference_latent_minus_direct', 'event_capture_difference_latent_minus_direct', 'p_fisher_event_rate_latent_gt_direct', 'p_proportion_event_rate_latent_gt_direct', 'mcnemar_b_event_latent_only', 'mcnemar_c_event_direct_only', 'p_mcnemar_event_capture_latent_gt_direct']
    print(res[show_cols].to_string(index=False))
    print('\nSaved results:')
    print(out_csv)
    print('\nHow to read:')
    print('  1) p_fisher_event_rate_latent_gt_direct:')
    print('     tests whether event rate is higher in latent segment than direct segment.')
    print('  2) p_mcnemar_event_capture_latent_gt_direct:')
    print('     tests among event cases only whether more events are captured by latent than direct.')
if __name__ == '__main__':
    main()


### Figure 4d — Two-sided risk-segment tests


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
warnings.filterwarnings('ignore')
BASE = '.'
IN_DIR = os.path.join(BASE, 'code_3_3_save/compare_external_mean_fivefold_fullcox_visit02_aligned')
PRED_DIR = os.path.join(IN_DIR, 'preds')
OUT_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_top_risk_segment_significance_2sided')
os.makedirs(OUT_DIR, exist_ok=True)
ENDPOINT = 'ICVD'
ENDPOINT_DISPLAY = 'ASCVD'
METHOD_FILES = {'Direct mean': 'external_direct_mean', 'latent-based multimodel aggregate': 'visit02_best_multimodal'}
METHOD_ORDER = ['Direct mean', 'latent-based multimodel aggregate']
DIRECT_METHOD = 'Direct mean'
LATENT_METHOD = 'latent-based multimodel aggregate'
SEGMENTS = [{'segment': 'top0_to_top5', 'display': 'Top 0-5%', 'low_pct': 0.95, 'high_pct': 1.000001}, {'segment': 'top5_to_top40', 'display': 'Top 5-40%', 'low_pct': 0.6, 'high_pct': 0.95}, {'segment': 'top5_to_top100', 'display': 'Top 5-100%', 'low_pct': -1e-06, 'high_pct': 0.95}]

def rank_percentile_high_is_high(x):
    s = pd.Series(x).astype(float)
    if len(s) <= 1:
        return np.zeros(len(s), dtype=float)
    r = s.rank(method='average').values
    return (r - 1) / (len(r) - 1 + 1e-08)

def get_pred_file(endpoint, method_key):
    return os.path.join(PRED_DIR, f'{endpoint}_{method_key}_external_aligned_pred.csv')

def load_one_method(endpoint, method_label, method_key):
    path = get_pred_file(endpoint, method_key)
    if not os.path.exists(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path)):
        raise FileNotFoundError(path)
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    if 'ID' in df.columns:
        df = df.rename(columns={'ID': 'pid'})
    required = ['pid', 'time', 'event', 'risk_norm']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(missing)
    df = df[required].copy()
    df['pid'] = pd.to_numeric(df['pid'], errors='coerce')
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df['event'] = pd.to_numeric(df['event'], errors='coerce').fillna(0).astype(int)
    df['risk_norm'] = pd.to_numeric(df['risk_norm'], errors='coerce')
    df = df.dropna()
    df['pid'] = df['pid'].astype(int)
    df = df.rename(columns={'risk_norm': f'{method_label}_risk'})
    return df

def load_data():
    dfs = []
    for m in METHOD_ORDER:
        df = load_one_method(ENDPOINT, m, METHOD_FILES[m])
        dfs.append(df)
    merged = dfs[0]
    for df in dfs[1:]:
        merged = merged.merge(df, on=['pid', 'time', 'event'], how='inner')
    merged['is_event'] = merged['event'].astype(int)
    for m in METHOD_ORDER:
        merged[f'{m}_pct'] = rank_percentile_high_is_high(merged[f'{m}_risk'].values)
    return merged

def get_mask(df, method, low, high):
    col = f'{method}_pct'
    return (df[col] >= low) & (df[col] < high)

def summarize(df, mask):
    sub = df[mask]
    n = len(sub)
    e = sub['is_event'].sum()
    ne = n - e
    return (n, e, ne)

def test_segment(df, seg):
    latent_mask = get_mask(df, LATENT_METHOD, seg['low_pct'], seg['high_pct'])
    direct_mask = get_mask(df, DIRECT_METHOD, seg['low_pct'], seg['high_pct'])
    (ln, le, lne) = summarize(df, latent_mask)
    (dn, de, dne) = summarize(df, direct_mask)
    table = np.array([[le, lne], [de, dne]])
    try:
        (_, p_fisher) = fisher_exact(table, alternative='two-sided')
    except:
        p_fisher = np.nan
    try:
        (_, p_z) = proportions_ztest(count=[le, de], nobs=[ln, dn], alternative='two-sided')
    except:
        p_z = np.nan
    event_df = df[df['is_event'] == 1]
    lm = latent_mask.loc[event_df.index]
    dm = direct_mask.loc[event_df.index]
    b = (lm & ~dm).sum()
    c = (~lm & dm).sum()
    n = b + c
    if n == 0:
        p_mcnemar = np.nan
    else:
        from scipy.stats import binomtest
        p_mcnemar = binomtest(b, n, 0.5, alternative='two-sided').pvalue
    return {'segment': seg['display'], 'latent_event_rate': le / ln if ln > 0 else np.nan, 'direct_event_rate': de / dn if dn > 0 else np.nan, 'latent_capture': le / df['is_event'].sum(), 'direct_capture': de / df['is_event'].sum(), 'p_fisher_two_sided': p_fisher, 'p_z_two_sided': p_z, 'p_mcnemar_two_sided': p_mcnemar}

def main():
    print('Loading data...')
    df = load_data()
    print('n:', len(df))
    print('event:', df['is_event'].sum())
    rows = []
    for seg in SEGMENTS:
        print('\nSegment:', seg['display'])
        res = test_segment(df, seg)
        rows.append(res)
        print('latent event rate:', res['latent_event_rate'])
        print('direct event rate:', res['direct_event_rate'])
        print('p_fisher:', res['p_fisher_two_sided'])
        print('p_mcnemar:', res['p_mcnemar_two_sided'])
    out = pd.DataFrame(rows)
    out_path = os.path.join(OUT_DIR, 'ICVD_segment_significance_two_sided.csv')
    out.to_csv(out_path, index=False)
    print('\nSaved:', out_path)
    print(out)
if __name__ == '__main__':
    main()


### Figure 4d — Event-rate and capture tests


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportions_ztest
warnings.filterwarnings('ignore')
BASE = '.'
IN_DIR = os.path.join(BASE, 'code_3_3_save/compare_external_mean_fivefold_fullcox_visit02_aligned')
PRED_DIR = os.path.join(IN_DIR, 'preds')
OUT_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_dual_metric_segment_tests')
os.makedirs(OUT_DIR, exist_ok=True)
ENDPOINT = 'ICVD'
METHOD_FILES = {'Direct mean': 'external_direct_mean', 'latent': 'visit02_best_multimodal'}
METHOD_ORDER = ['Direct mean', 'latent']
DIRECT = 'Direct mean'
LATENT = 'latent'
SEGMENTS = [{'name': 'Top 0-5%', 'low': 0.95, 'high': 1.000001}, {'name': 'Top 5-40%', 'low': 0.6, 'high': 0.95}, {'name': 'Top 5-100%', 'low': -1e-06, 'high': 0.95}]

def rank_pct(x):
    s = pd.Series(x).astype(float)
    r = s.rank(method='average').values
    return (r - 1) / (len(r) - 1 + 1e-09)

def load_one(method, key):
    path = os.path.join(PRED_DIR, f'{ENDPOINT}_{key}_external_aligned_pred.csv')
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    if 'ID' in df.columns:
        df = df.rename(columns={'ID': 'pid'})
    df = df[['pid', 'time', 'event', 'risk_norm']].copy()
    df['event'] = df['event'].fillna(0).astype(int)
    df = df.rename(columns={'risk_norm': f'{method}_risk'})
    return df

def load_data():
    dfs = []
    for m in METHOD_ORDER:
        dfs.append(load_one(m, METHOD_FILES[m]))
    df = dfs[0]
    for d in dfs[1:]:
        df = df.merge(d, on=['pid', 'time', 'event'])
    df['is_event'] = df['event']
    for m in METHOD_ORDER:
        df[f'{m}_pct'] = rank_pct(df[f'{m}_risk'])
    return df

def mask(df, method, low, high):
    return (df[f'{method}_pct'] >= low) & (df[f'{method}_pct'] < high)

def test_segment(df, seg):
    latent_m = mask(df, LATENT, seg['low'], seg['high'])
    direct_m = mask(df, DIRECT, seg['low'], seg['high'])
    total_events = df['is_event'].sum()
    ln = latent_m.sum()
    dn = direct_m.sum()
    le = df.loc[latent_m, 'is_event'].sum()
    de = df.loc[direct_m, 'is_event'].sum()
    latent_event_rate = le / ln if ln > 0 else np.nan
    direct_event_rate = de / dn if dn > 0 else np.nan
    table_rate = np.array([[le, ln - le], [de, dn - de]])
    try:
        (_, p_rate) = fisher_exact(table_rate, alternative='two-sided')
    except:
        p_rate = np.nan
    latent_capture = le / total_events if total_events > 0 else np.nan
    direct_capture = de / total_events if total_events > 0 else np.nan
    table_capture = np.array([[le, total_events - le], [de, total_events - de]])
    try:
        (_, p_capture) = fisher_exact(table_capture, alternative='two-sided')
    except:
        p_capture = np.nan
    try:
        (_, p_z_capture) = proportions_ztest(count=[le, de], nobs=[total_events, total_events], alternative='two-sided')
    except:
        p_z_capture = np.nan
    event_df = df[df['is_event'] == 1]
    lm = latent_m.loc[event_df.index]
    dm = direct_m.loc[event_df.index]
    b = (lm & ~dm).sum()
    c = (~lm & dm).sum()
    try:
        from scipy.stats import binomtest
        p_mcnemar = binomtest(b, b + c, 0.5, alternative='two-sided').pvalue
    except:
        p_mcnemar = np.nan
    return {'segment': seg['name'], 'latent_event_rate': latent_event_rate, 'direct_event_rate': direct_event_rate, 'p_rate': p_rate, 'latent_capture': latent_capture, 'direct_capture': direct_capture, 'p_capture': p_capture, 'p_z_capture': p_z_capture, 'p_mcnemar_capture': p_mcnemar}

def main():
    df = load_data()
    print('n:', len(df))
    print('events:', df['is_event'].sum())
    results = []
    for seg in SEGMENTS:
        print('\nSegment:', seg['name'])
        r = test_segment(df, seg)
        results.append(r)
        print('rate p:', r['p_rate'])
        print('capture p:', r['p_capture'])
        print('mcnemar:', r['p_mcnemar_capture'])
    out = pd.DataFrame(results)
    out_path = os.path.join(OUT_DIR, 'ICVD_dual_metric_segment_results.csv')
    out.to_csv(out_path, index=False)
    print('\nSaved:', out_path)
    print(out)
if __name__ == '__main__':
    main()


### Figure 4e — Protein pathway enrichment


In [ ]:
import os
import re
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import gseapy as gp
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE = '.'
ENDPOINT = 'ICVD'
ENDPOINT_LABEL = 'ASCVD'
DIFF_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_protein_multimodal_top5_vs_direct_mean_top5_FDR_BH_and_Storey')
GSEA_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_protein_top5_boxplot_and_groupwise_GSEA')
DIFF_TABLE = os.path.join(DIFF_DIR, 'ICVD_protein_multimodal_top5_vs_direct_mean_top5_all_diff.csv')
GSEA_NES_TABLE = os.path.join(GSEA_DIR, 'ICVD_GSEA_groupwise_NES_change_multimodal_minus_direct.csv')
OUT_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_GO_BP_FDR005_and_selected_GSEA_NES_subplots')
os.makedirs(OUT_DIR, exist_ok=True)
FDR_FOR_GENE_SELECTION = 'fdr_bh'
PROTEIN_FDR_CUTOFF = 0.05
GO_FDR_CUTOFF = 0.05
TOP_N_GO_TERMS = 8
PLOT_TOP_GO_IF_NO_SIGNIFICANT = False
SELECTED_PATHWAYS = ['Fatty Acid Metabolism', 'Angiogenesis', 'Cholesterol Homeostasis', 'Complement', 'Coagulation', 'Hypoxia', 'IL-2/STAT5 Signaling', 'Epithelial Mesenchymal Transition']
COLOR_DIRECT = '#a2a8a5'
COLOR_MULTI = '#e4937d'
COLOR_GO_HIGHER = '#e4937d'
COLOR_GO_LOWER = '#a2a8a5'
DIRECTION_COLORS = {'higher_in_multimodal_top5': COLOR_GO_HIGHER, 'lower_in_multimodal_top5': COLOR_GO_LOWER}

def clean_gene(x):
    x = str(x)
    x = re.sub('\\.\\d+$', '', x)
    if x.startswith('seq.'):
        parts = x.split('.')
        if len(parts) >= 2:
            x = parts[1]
    return x.upper()

def clean_name(x):
    x = str(x)
    x = x.strip()
    x = re.sub('[^A-Za-z0-9_.+-]+', '_', x)
    return x

def normalize_term(x):
    x = str(x)
    x = x.replace('HALLMARK_', '')
    x = x.replace('_', ' ')
    x = x.replace('-', ' ')
    x = re.sub('\\s+', ' ', x)
    return x.strip().lower()

def pretty_pathway(x):
    x = str(x)
    x = x.replace('HALLMARK_', '')
    x = x.replace('_', ' ')
    x = re.sub('\\s+', ' ', x)
    return x.strip().title()

def shorten_go_term(x):
    x = str(x)
    x = re.sub('\\s*\\(GO:\\d+\\)', '', x)
    x = x.replace('_', ' ')
    x = x.strip().lower()
    words = x.split()
    if len(words) <= 3:
        return x
    lines = []
    cur = []
    for w in words:
        cur.append(w)
        if len(' '.join(cur)) >= 18:
            lines.append(' '.join(cur))
            cur = []
    if len(cur) > 0:
        lines.append(' '.join(cur))
    return '\n'.join(lines)

def get_adjusted_p_col(df):
    candidates = ['Adjusted P-value', 'Adjusted_P-value', 'Adjusted.P.value', 'FDR', 'fdr', 'qvalue', 'q_value']
    for c in candidates:
        if c in df.columns:
            return c
    return None

def get_gene_col(df):
    candidates = ['Genes', 'genes', 'Overlap genes', 'Overlap Genes', 'overlap_genes']
    for c in candidates:
        if c in df.columns:
            return c
    return None

def standardize_enrichr_result(df):
    out = df.copy()
    if 'Term' not in out.columns:
        raise ValueError('Enrichr result missing Term column.')
    fdr_col = get_adjusted_p_col(out)
    if fdr_col is None:
        raise ValueError('Cannot find adjusted P / FDR column in Enrichr result.')
    gene_col = get_gene_col(out)
    if gene_col is None:
        out['genes'] = ''
    else:
        out['genes'] = out[gene_col].astype(str)
    out['fdr'] = pd.to_numeric(out[fdr_col], errors='coerce')
    if 'P-value' in out.columns:
        out['pvalue'] = pd.to_numeric(out['P-value'], errors='coerce')
    else:
        out['pvalue'] = np.nan
    out['neglog10_fdr'] = -np.log10(out['fdr'].clip(lower=1e-300))
    return out

def load_diff_table():
    if not os.path.exists(str(OUTPUT_ROOT / DIFF_TABLE if (OUTPUT_ROOT / DIFF_TABLE).exists() else INPUT_ROOT / DIFF_TABLE)):
        raise FileNotFoundError(f'Cannot find protein differential table:\n{DIFF_TABLE}')
    df = pd.read_csv(str(OUTPUT_ROOT / DIFF_TABLE if (OUTPUT_ROOT / DIFF_TABLE).exists() else INPUT_ROOT / DIFF_TABLE))
    required = ['feature', 'gene', 'direction', 'p', FDR_FOR_GENE_SELECTION]
    missing = [c for c in required if c not in df.columns]
    if len(missing) > 0:
        raise ValueError(f'Missing columns in differential table: {missing}\nAvailable columns:\n{df.columns.tolist()}')
    df['gene'] = df['gene'].map(clean_gene)
    df[FDR_FOR_GENE_SELECTION] = pd.to_numeric(df[FDR_FOR_GENE_SELECTION], errors='coerce')
    return df

def get_sig_genes_by_direction(diff_df, direction):
    sub = diff_df[(diff_df['direction'] == direction) & (diff_df[FDR_FOR_GENE_SELECTION] < PROTEIN_FDR_CUTOFF)].copy()
    sub = sub.dropna(subset=['gene', FDR_FOR_GENE_SELECTION]).copy()
    genes = sub['gene'].astype(str).str.upper().replace('', np.nan).dropna().unique().tolist()
    genes = sorted(set(genes))
    return (genes, sub)

def run_go_bp_enrichment(genes, direction):
    if len(genes) < 3:
        print(f'[WARN] Too few genes for GO BP enrichment: {direction}, n={len(genes)}')
        return pd.DataFrame()
    print('\nRunning GO Biological Process enrichment')
    print('  direction:', direction)
    print('  n genes:', len(genes))
    try:
        enr = gp.enrichr(gene_list=genes, gene_sets='GO_Biological_Process_2023', organism='Human', outdir=None, cutoff=1.0, no_plot=True)
        res = enr.results.copy()
    except Exception as e:
        print(f'[WARN] GO BP enrichment failed for {direction}: {e}')
        return pd.DataFrame()
    if len(res) == 0:
        return res
    res = standardize_enrichr_result(res)
    res['endpoint'] = ENDPOINT
    res['endpoint_label'] = ENDPOINT_LABEL
    res['direction'] = direction
    res['protein_fdr_used'] = FDR_FOR_GENE_SELECTION
    res['protein_fdr_cutoff'] = PROTEIN_FDR_CUTOFF
    res['go_fdr_cutoff'] = GO_FDR_CUTOFF
    res['input_gene_count'] = len(genes)
    res['input_genes'] = ';'.join(genes)
    res = res.sort_values(['fdr', 'pvalue'], ascending=[True, True]).reset_index(drop=True)
    out_csv = os.path.join(OUT_DIR, f'{ENDPOINT}_GO_BP_enrichment_{direction}_{FDR_FOR_GENE_SELECTION}_lt_{PROTEIN_FDR_CUTOFF:.2f}.csv')
    res.to_csv(out_csv, index=False)
    print('Saved GO BP enrichment:', out_csv)
    return res

def plot_go_bp_barplot(go_df, direction):
    if len(go_df) == 0:
        print(f'[WARN] No GO BP result for {direction}.')
        return
    sig = go_df[go_df['fdr'] < GO_FDR_CUTOFF].copy()
    if len(sig) == 0 and PLOT_TOP_GO_IF_NO_SIGNIFICANT:
        sig = go_df.sort_values(['fdr', 'pvalue'], ascending=[True, True]).head(TOP_N_GO_TERMS).copy()
    else:
        sig = sig.sort_values(['fdr', 'pvalue'], ascending=[True, True]).head(TOP_N_GO_TERMS).copy()
    if len(sig) == 0:
        print(f'[WARN] No GO BP terms with FDR < {GO_FDR_CUTOFF} for {direction}.')
        return
    sig = sig.iloc[::-1].copy()
    sig['term_label'] = sig['Term'].map(shorten_go_term)
    sig['gene_label'] = sig['genes'].astype(str).str.replace(';', ', ', regex=False).str.replace('/', ', ', regex=False)
    color = DIRECTION_COLORS.get(direction, COLOR_GO_HIGHER)
    if direction == 'higher_in_multimodal_top5':
        title = 'Higher in multimodal top5%'
    elif direction == 'lower_in_multimodal_top5':
        title = 'Lower in multimodal top5%\nHigher in direct mean top5%'
    else:
        title = direction
    fig_h = max(3.8, 0.82 * len(sig) + 1.0)
    (fig, ax) = plt.subplots(figsize=(7.8, fig_h))
    y = np.arange(len(sig))
    x = sig['neglog10_fdr'].values
    ax.barh(y, x, color=color, alpha=0.92, height=0.55)
    xmax = max(1.0, np.nanmax(x))
    for (i, (_, row)) in enumerate(sig.iterrows()):
        gene_label = str(row['gene_label'])
        if len(gene_label) > 48:
            gene_label = gene_label[:45] + '...'
        ax.text(0.04 * xmax, i, gene_label, va='center', ha='left', fontsize=9, color='black')
    ax.set_yticks(y)
    ax.set_yticklabels(sig['term_label'], fontsize=11, color='#4a4a4a')
    ax.set_xlabel('$-\\log_{10}(\\mathrm{FDR})$', fontsize=13, color='#4a4a4a')
    ax.set_title(title, fontsize=18, color='#4a4a4a', loc='left', pad=10)
    ax.axvline(0, color='#D9D9D9', linewidth=2)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#D9D9D9')
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_color('#D9D9D9')
    ax.spines['bottom'].set_linewidth(2)
    ax.tick_params(axis='x', labelsize=11, colors='#4a4a4a')
    ax.tick_params(axis='y', length=0)
    ax.set_xlim(0, max(3.0, np.nanmax(x) * 1.1))
    plt.tight_layout()
    out_pdf = os.path.join(OUT_DIR, f'{ENDPOINT}_GO_BP_barplot_{direction}_{FDR_FOR_GENE_SELECTION}_lt_{PROTEIN_FDR_CUTOFF:.2f}_GOFDR005.pdf')
    out_png = os.path.join(OUT_DIR, f'{ENDPOINT}_GO_BP_barplot_{direction}_{FDR_FOR_GENE_SELECTION}_lt_{PROTEIN_FDR_CUTOFF:.2f}_GOFDR005.png')
    plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved GO BP barplot:', out_pdf)
    print('Saved GO BP barplot:', out_png)

def run_and_plot_go_bp(diff_df):
    all_go = []
    for direction in ['higher_in_multimodal_top5', 'lower_in_multimodal_top5']:
        (genes, sig_df) = get_sig_genes_by_direction(diff_df, direction=direction)
        gene_out = os.path.join(OUT_DIR, f'{ENDPOINT}_GO_BP_input_genes_{direction}_{FDR_FOR_GENE_SELECTION}_lt_{PROTEIN_FDR_CUTOFF:.2f}.csv')
        sig_df.to_csv(gene_out, index=False)
        print('\nSaved GO input gene table:', gene_out)
        print('Direction:', direction)
        print('n genes:', len(genes))
        print('genes:', ', '.join(genes[:40]))
        go_df = run_go_bp_enrichment(genes=genes, direction=direction)
        if len(go_df) > 0:
            all_go.append(go_df)
            plot_go_bp_barplot(go_df=go_df, direction=direction)
    if len(all_go) > 0:
        merged = pd.concat(all_go, axis=0, ignore_index=True, sort=False)
        out_csv = os.path.join(OUT_DIR, f'{ENDPOINT}_GO_BP_enrichment_all_directions_{FDR_FOR_GENE_SELECTION}_lt_{PROTEIN_FDR_CUTOFF:.2f}_GOFDR005.csv')
        merged.to_csv(out_csv, index=False)
        print('Saved merged GO BP enrichment:', out_csv)
        return merged
    return pd.DataFrame()

def load_gsea_nes_table():
    if not os.path.exists(str(OUTPUT_ROOT / GSEA_NES_TABLE if (OUTPUT_ROOT / GSEA_NES_TABLE).exists() else INPUT_ROOT / GSEA_NES_TABLE)):
        raise FileNotFoundError(f'Cannot find GSEA NES table:\n{GSEA_NES_TABLE}')
    df = pd.read_csv(str(OUTPUT_ROOT / GSEA_NES_TABLE if (OUTPUT_ROOT / GSEA_NES_TABLE).exists() else INPUT_ROOT / GSEA_NES_TABLE))
    required = ['Term', 'NES_multimodal_top5', 'NES_direct_mean_top5', 'NES_change_multimodal_minus_direct']
    missing = [c for c in required if c not in df.columns]
    if len(missing) > 0:
        raise ValueError(f'GSEA table missing columns: {missing}\nAvailable columns:\n{df.columns.tolist()}')
    df['term_norm'] = df['Term'].map(normalize_term)
    return df

def subset_selected_gsea(gsea_df):
    rows = []
    for pathway in SELECTED_PATHWAYS:
        p_norm = normalize_term(pathway)
        tmp = gsea_df[gsea_df['term_norm'] == p_norm].copy()
        if len(tmp) == 0:
            tmp = gsea_df[gsea_df['term_norm'].str.contains(p_norm, regex=False, na=False)].copy()
        if len(tmp) == 0:
            print('[WARN] selected pathway not found:', pathway)
            rows.append({'pathway': pathway, 'Term': pathway, 'NES_multimodal_top5': np.nan, 'NES_direct_mean_top5': np.nan, 'NES_change_multimodal_minus_direct': np.nan, 'qvalue_multimodal_top5': np.nan, 'qvalue_direct_mean_top5': np.nan})
            continue
        r = tmp.iloc[0].to_dict()
        r['pathway'] = pathway
        rows.append(r)
    out = pd.DataFrame(rows)
    return out

def plot_selected_gsea_one_row(selected_df):
    selected_out = os.path.join(OUT_DIR, f'{ENDPOINT}_selected_GSEA_NES_values_one_row.csv')
    selected_df.to_csv(selected_out, index=False)
    print('Saved selected GSEA NES values:', selected_out)
    n_path = len(SELECTED_PATHWAYS)
    (fig, axes) = plt.subplots(1, n_path, figsize=(2.15 * n_path, 4.6), sharey=False)
    if n_path == 1:
        axes = [axes]
    all_vals = []
    for c in ['NES_multimodal_top5', 'NES_direct_mean_top5']:
        vals = pd.to_numeric(selected_df[c], errors='coerce').values
        all_vals.extend(list(np.abs(vals[np.isfinite(vals)])))
    max_abs = max(all_vals) if len(all_vals) > 0 else 1.0
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0
    xlim = max_abs * 1.35
    for (ax, pathway) in zip(axes, SELECTED_PATHWAYS):
        sub = selected_df[selected_df['pathway'] == pathway].copy()
        if len(sub) == 0:
            ax.axis('off')
            continue
        row = sub.iloc[0]
        nes_multi = pd.to_numeric(row.get('NES_multimodal_top5', np.nan), errors='coerce')
        nes_direct = pd.to_numeric(row.get('NES_direct_mean_top5', np.nan), errors='coerce')
        x_direct = -abs(nes_direct) if np.isfinite(nes_direct) else np.nan
        x_multi = abs(nes_multi) if np.isfinite(nes_multi) else np.nan
        ax.axvline(0, color='#D9D9D9', linewidth=2, zorder=0)
        if np.isfinite(x_direct):
            ax.barh(0, x_direct, color=COLOR_DIRECT, alpha=0.95, height=0.34)
            ax.text(x_direct - 0.04 * xlim, 0, f'{nes_direct:.2f}', va='center', ha='right', fontsize=9, color='black')
        if np.isfinite(x_multi):
            ax.barh(0, x_multi, color=COLOR_MULTI, alpha=0.95, height=0.34)
            ax.text(x_multi + 0.04 * xlim, 0, f'{nes_multi:.2f}', va='center', ha='left', fontsize=9, color='black')
        ax.set_xlim(-xlim, xlim)
        ax.set_ylim(-0.75, 0.75)
        ax.set_yticks([])
        title = pretty_pathway(pathway)
        title_words = title.split()
        if len(title_words) > 2:
            mid = int(np.ceil(len(title_words) / 2))
            title = ' '.join(title_words[:mid]) + '\n' + ' '.join(title_words[mid:])
        ax.set_title(title, fontsize=11, color='#4a4a4a', pad=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_color('#D9D9D9')
        ax.spines['bottom'].set_linewidth(1.6)
        ax.tick_params(axis='x', labelsize=9, colors='#4a4a4a')
        ax.grid(axis='x', linestyle='--', alpha=0.16)
    fig.text(0.28, 0.02, 'Direct mean top5%', ha='center', va='center', fontsize=13, color='#4a4a4a')
    fig.text(0.72, 0.02, 'Multimodal top5%', ha='center', va='center', fontsize=13, color='#4a4a4a')
    fig.suptitle(f'{ENDPOINT_LABEL}: selected pathway NES in top5% groups', fontsize=18, color='#4a4a4a', y=1.03)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    out_pdf = os.path.join(OUT_DIR, f'{ENDPOINT}_selected_GSEA_NES_one_row_8pathways_direct_left_multimodal_right.pdf')
    out_png = os.path.join(OUT_DIR, f'{ENDPOINT}_selected_GSEA_NES_one_row_8pathways_direct_left_multimodal_right.png')
    plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved selected GSEA NES one-row plot:', out_pdf)
    print('Saved selected GSEA NES one-row plot:', out_png)

def main():
    print('\n================================================')
    print(f'Endpoint: {ENDPOINT} ({ENDPOINT_LABEL})')
    print('1. GO BP enrichment with GO FDR < 0.05')
    print('2. Selected GSEA NES one-row subplot plot')
    print('================================================')
    print('\nInput differential table:')
    print(DIFF_TABLE)
    diff_df = load_diff_table()
    print('Loaded differential table')
    print('  rows:', len(diff_df))
    print('  protein FDR used for gene selection:', FDR_FOR_GENE_SELECTION)
    print('  protein FDR cutoff:', PROTEIN_FDR_CUTOFF)
    print('  GO FDR cutoff:', GO_FDR_CUTOFF)
    print('\n================================================')
    print('GO Biological Process enrichment')
    print('================================================')
    go_all = run_and_plot_go_bp(diff_df)
    print('\n================================================')
    print('Selected GSEA NES plot')
    print('================================================')
    print('Input GSEA table:')
    print(GSEA_NES_TABLE)
    gsea_df = load_gsea_nes_table()
    selected_df = subset_selected_gsea(gsea_df)
    plot_selected_gsea_one_row(selected_df)
    print('\n================================================')
    print('Done.')
    print('================================================')
    print('Output dir:', OUT_DIR)
    print('\nKey outputs:')
    print(f'  {ENDPOINT}_GO_BP_enrichment_all_directions_{FDR_FOR_GENE_SELECTION}_lt_{PROTEIN_FDR_CUTOFF:.2f}_GOFDR005.csv')
    print(f'  {ENDPOINT}_GO_BP_barplot_higher_in_multimodal_top5_{FDR_FOR_GENE_SELECTION}_lt_{PROTEIN_FDR_CUTOFF:.2f}_GOFDR005.pdf')
    print(f'  {ENDPOINT}_GO_BP_barplot_lower_in_multimodal_top5_{FDR_FOR_GENE_SELECTION}_lt_{PROTEIN_FDR_CUTOFF:.2f}_GOFDR005.pdf')
    print(f'  {ENDPOINT}_selected_GSEA_NES_values_one_row.csv')
    print(f'  {ENDPOINT}_selected_GSEA_NES_one_row_8pathways_direct_left_multimodal_right.pdf')
    print('\nInterpretation:')
    print('  GO BP:')
    print('      Only GO terms with enrichment FDR < 0.05 are plotted.')
    print('  GSEA selected pathway plot:')
    print('      Each pathway is one subplot.')
    print('      Left gray bar = Direct mean top5% NES.')
    print('      Right orange bar = Multimodal top5% NES.')
    print('      Bar length = abs(NES); text label = actual NES value.')
if __name__ == '__main__':
    main()


### Figure 4e — Selected GSEA pathways


In [ ]:
import os
import re
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE = '.'
ENDPOINT = 'ICVD'
ENDPOINT_LABEL = 'ASCVD'
GSEA_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_protein_top5_boxplot_and_groupwise_GSEA')
GSEA_NES_TABLE = os.path.join(GSEA_DIR, 'ICVD_GSEA_groupwise_NES_change_multimodal_minus_direct.csv')
OUT_DIR = os.path.join(BASE, 'fig4_final', 'ICVD_selected_GSEA_NES_barplots')
os.makedirs(OUT_DIR, exist_ok=True)
SELECTED_PATHWAYS = ['Fatty Acid Metabolism', 'Angiogenesis', 'Cholesterol Homeostasis', 'Complement', 'Coagulation', 'Hypoxia', 'IL-2/STAT5 Signaling', 'Epithelial Mesenchymal Transition']
MODEL_COLORS = {'Direct mean': '#a2a8a5', 'Multimodal': '#e4937d'}
MODEL_ORDER = ['Direct mean', 'Multimodal']
N_COLS = 4
FIG_W_PER_PANEL = 1.6
FIG_H_PER_PANEL = 3.2
SAVE_PREFIX = 'ICVD_selected_GSEA_NES_direct_mean_vs_multimodal'

def normalize_term(x):
    x = str(x)
    x = x.replace('HALLMARK_', '')
    x = x.replace('_', ' ')
    x = x.replace('-', ' ')
    x = re.sub('\\s+', ' ', x)
    return x.strip().lower()

def pretty_pathway(x):
    x = str(x)
    x = x.replace('HALLMARK_', '')
    x = x.replace('_', ' ')
    x = re.sub('\\s+', ' ', x)
    x = x.strip()
    return x.title()

def wrap_title(x, max_words_per_line=3):
    words = str(x).split()
    if len(words) <= max_words_per_line:
        return str(x)
    lines = []
    for i in range(0, len(words), max_words_per_line):
        lines.append(' '.join(words[i:i + max_words_per_line]))
    return '\n'.join(lines)

def load_gsea_table():
    if not os.path.exists(str(OUTPUT_ROOT / GSEA_NES_TABLE if (OUTPUT_ROOT / GSEA_NES_TABLE).exists() else INPUT_ROOT / GSEA_NES_TABLE)):
        raise FileNotFoundError(f'Cannot find GSEA NES table:\n{GSEA_NES_TABLE}')
    df = pd.read_csv(str(OUTPUT_ROOT / GSEA_NES_TABLE if (OUTPUT_ROOT / GSEA_NES_TABLE).exists() else INPUT_ROOT / GSEA_NES_TABLE))
    required = ['Term', 'NES_multimodal_top5', 'NES_direct_mean_top5', 'NES_change_multimodal_minus_direct']
    missing = [c for c in required if c not in df.columns]
    if len(missing) > 0:
        raise ValueError(f'GSEA table missing columns: {missing}\nAvailable columns:\n{df.columns.tolist()}')
    df['term_norm'] = df['Term'].map(normalize_term)
    return df

def subset_selected_pathways(gsea_df):
    rows = []
    for pathway in SELECTED_PATHWAYS:
        p_norm = normalize_term(pathway)
        tmp = gsea_df[gsea_df['term_norm'] == p_norm].copy()
        if len(tmp) == 0:
            tmp = gsea_df[gsea_df['term_norm'].str.contains(p_norm, regex=False, na=False)].copy()
        if len(tmp) == 0:
            print('[WARN] selected pathway not found:', pathway)
            rows.append({'pathway': pathway, 'Term': pathway, 'NES_direct_mean_top5': np.nan, 'NES_multimodal_top5': np.nan, 'NES_change_multimodal_minus_direct': np.nan, 'qvalue_direct_mean_top5': np.nan, 'qvalue_multimodal_top5': np.nan})
            continue
        r = tmp.iloc[0].to_dict()
        r['pathway'] = pathway
        rows.append(r)
    selected = pd.DataFrame(rows)
    return selected

def make_long_plot_df(selected_df):
    rows = []
    for (_, r) in selected_df.iterrows():
        pathway = r['pathway']
        rows.append({'pathway': pathway, 'model': 'Direct mean', 'NES': pd.to_numeric(r.get('NES_direct_mean_top5', np.nan), errors='coerce'), 'qvalue': pd.to_numeric(r.get('qvalue_direct_mean_top5', np.nan), errors='coerce')})
        rows.append({'pathway': pathway, 'model': 'Multimodal', 'NES': pd.to_numeric(r.get('NES_multimodal_top5', np.nan), errors='coerce'), 'qvalue': pd.to_numeric(r.get('qvalue_multimodal_top5', np.nan), errors='coerce')})
    plot_df = pd.DataFrame(rows)
    return plot_df

def plot_selected_gsea_barplots(selected_df):
    plot_df = make_long_plot_df(selected_df)
    values_out = os.path.join(OUT_DIR, f'{SAVE_PREFIX}_values.csv')
    selected_df.to_csv(values_out, index=False)
    long_out = os.path.join(OUT_DIR, f'{SAVE_PREFIX}_long_values.csv')
    plot_df.to_csv(long_out, index=False)
    print('Saved selected GSEA values:', values_out)
    print('Saved selected GSEA long values:', long_out)
    n = len(SELECTED_PATHWAYS)
    n_cols = min(N_COLS, n)
    n_rows = int(np.ceil(n / n_cols))
    fig_w = FIG_W_PER_PANEL * n_cols
    fig_h = FIG_H_PER_PANEL * n_rows
    (fig, axes) = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False, sharey=False)
    axes = axes.reshape(-1)
    all_nes = pd.to_numeric(plot_df['NES'], errors='coerce')
    max_abs = np.nanmax(np.abs(all_nes.values))
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0
    ylim = max_abs * 1.25
    for (ax_i, pathway) in enumerate(SELECTED_PATHWAYS):
        ax = axes[ax_i]
        sub = plot_df[plot_df['pathway'] == pathway].copy()
        sub['model'] = pd.Categorical(sub['model'], categories=MODEL_ORDER, ordered=True)
        sub = sub.sort_values('model')
        x = np.arange(len(MODEL_ORDER))
        for (i, model) in enumerate(MODEL_ORDER):
            tmp = sub[sub['model'] == model]
            if len(tmp) == 0:
                continue
            nes = tmp['NES'].values[0]
            if not np.isfinite(nes):
                continue
            ax.bar(i, nes, color=MODEL_COLORS[model], alpha=0.95, width=0.75)
            if nes >= 0:
                va = 'bottom'
                y_text = nes + 0.04 * ylim
            else:
                va = 'top'
                y_text = nes - 0.04 * ylim
            ax.text(i, y_text, f'{nes:.2f}', ha='center', va=va, fontsize=8, color='black')
        ax.axhline(0, linestyle='--', color='gray', linewidth=1.0, alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_ORDER, rotation=45, ha='right', fontsize=9)
        ax.set_ylim(-ylim, ylim)
        ax.set_title(wrap_title(pretty_pathway(pathway), max_words_per_line=3).replace(' ', '\n'), fontsize=12, pad=8)
        if ax_i % n_cols == 0:
            ax.set_ylabel('NES', fontsize=11)
        else:
            ax.set_ylabel('')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='y', linestyle='--', alpha=0.2)
    for j in range(len(SELECTED_PATHWAYS), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f'{ENDPOINT_LABEL}: selected pathway enrichment in top5% groups', fontsize=16, y=1.02)
    plt.tight_layout()
    out_pdf = os.path.join(OUT_DIR, f'{SAVE_PREFIX}.pdf')
    out_png = os.path.join(OUT_DIR, f'{SAVE_PREFIX}.png')
    plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved GSEA barplot:', out_pdf)
    print('Saved GSEA barplot:', out_png)

def main():
    print('\n================================================')
    print(f'Endpoint: {ENDPOINT} ({ENDPOINT_LABEL})')
    print('Selected GSEA NES barplots')
    print('================================================')
    print('Input GSEA NES table:')
    print(GSEA_NES_TABLE)
    gsea_df = load_gsea_table()
    selected_df = subset_selected_pathways(gsea_df)
    plot_selected_gsea_barplots(selected_df)
    print('\nDone.')
    print('Output dir:', OUT_DIR)
    print('\nKey outputs:')
    print(f'  {SAVE_PREFIX}.pdf')
    print(f'  {SAVE_PREFIX}.png')
    print(f'  {SAVE_PREFIX}_values.csv')
    print(f'  {SAVE_PREFIX}_long_values.csv')
    print('\nInterpretation:')
    print('  Each subplot is one pathway.')
    print('  Gray bar = Direct mean top5% groupwise protein-risk GSEA NES.')
    print('  Orange bar = Multimodal top5% groupwise protein-risk GSEA NES.')
    print('  NES > 0 means pathway genes tend to have positive protein-risk correlations.')
    print('  NES < 0 means pathway genes tend to have negative protein-risk correlations.')
if __name__ == '__main__':
    main()


### Supporting analysis for Figure 4f — Diffusion-map embedding views


In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import eigsh
from scipy.stats import rankdata, gaussian_kde
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
IN_DIR = Path('fig3/code_save_6_7')
OUT_DIR = Path('fig4_final/embedding_v2')
OUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_PATH = IN_DIR / 'sample_embeddings_ICVD_test.npy'
ATTN_PATH = IN_DIR / 'sample_attn_ICVD_test.npy'
RISK_PATH = IN_DIR / 'sample_risk_ICVD_test.npy'
AGE_PATH = 'data/private/data_2klh_longitudinal.tsv'
MODALITY_NAMES = ['ECG', 'Soma', 'SNP', 'Fundus']
TARGET_VISIT = '12'
N_NEIGHBORS = 30
N_COMPONENTS = 6
DIFFUSION_TIME = 1
RANDOM_STATE = 42
POINT_SIZE = 4
POINT_ALPHA = 0.65
DPI = 320
PLOT_TAG = 'ICVD_visit12_4modality_diffmap'
SAVE_FOURPANEL = True

def build_output_path(filename: str, out_dir: Path=OUT_DIR) -> str:
    out_dir.mkdir(parents=True, exist_ok=True)
    (stem, ext) = os.path.splitext(filename)
    candidate = out_dir / filename
    if not candidate.exists():
        return str(candidate)
    i = 1
    while True:
        candidate = out_dir / f'{stem}_v{i:02d}{ext}'
        if not candidate.exists():
            return str(candidate)
        i += 1

def load_npy_any(path: Path):
    obj = np.load(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path), allow_pickle=True)
    if isinstance(obj, np.ndarray) and obj.shape == () and (obj.dtype == object):
        return obj.item()
    return obj

def clean_id(x):
    x = pd.Series(x).astype(str)
    x = x.str.replace('.0', '', regex=False)
    x = x.str.strip()
    return x

def to_float_or_nan(x):
    try:
        if x is None:
            return np.nan
        if isinstance(x, (list, tuple, np.ndarray)):
            arr = np.array(x, dtype=object).reshape(-1)
            if len(arr) == 1:
                x = arr[0]
            else:
                return np.nan
        return float(x)
    except Exception:
        return np.nan

def parse_sample_id(sample_key):
    s = str(sample_key)
    if '-' in s:
        (pid, visit) = s.split('-', 1)
        return (str(pid), str(visit))
    return (str(s), 'UNK')

def rank_norm(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)
    out = np.full(len(x), np.nan, dtype=float)
    if ok.sum() == 0:
        return out
    r = rankdata(x[ok], method='average')
    if len(r) == 1:
        out[ok] = 0.5
    else:
        out[ok] = (r - 1) / (len(r) - 1)
    return out

def clip_percentile(x, p_low=2, p_high=98):
    (lo, hi) = np.nanpercentile(x, [p_low, p_high])
    return (np.clip(x, lo, hi), lo, hi)

def density_alpha(x, y):
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)
    z = (z - z.min()) / (z.max() - z.min() + 1e-08)
    return 0.2 + 0.6 * (1 - z)

def extract_meta_from_value(v):
    risk = np.nan
    event = np.nan
    time = np.nan
    if isinstance(v, dict):
        for k in ['risk', 'risk_norm', 'pred', 'score', 'rank', 'yhat', 'risk_raw', 'prediction', 'prob']:
            if k in v:
                risk = to_float_or_nan(v.get(k))
                if np.isfinite(risk):
                    break
        if not np.isfinite(risk):
            for val in v.values():
                num = to_float_or_nan(val)
                if np.isfinite(num):
                    risk = num
                    break
        for k in ['event', 'event_binary', 'label', 'y', 'outcome', 'case', 'status']:
            if k in v:
                event = to_float_or_nan(v.get(k))
                if np.isfinite(event):
                    break
        for k in ['time', 'tte', 'time_to_event', 'surv_time', 'followup_time', 'duration']:
            if k in v:
                time = to_float_or_nan(v.get(k))
                if np.isfinite(time):
                    break
    elif isinstance(v, (list, tuple, np.ndarray)):
        arr = np.array(v, dtype=object).reshape(-1)
        if len(arr) > 0:
            risk = to_float_or_nan(arr[0])
        if len(arr) > 1:
            event = to_float_or_nan(arr[1])
        if len(arr) > 2:
            time = to_float_or_nan(arr[2])
    else:
        risk = to_float_or_nan(v)
    return (risk, event, time)

def infer_sample_order_from_embedding(emb_obj):
    if isinstance(emb_obj, dict):
        sample_ids = list(emb_obj.keys())
        emb_list = []
        for sid in sample_ids:
            emb_list.append(np.asarray(emb_obj[sid], dtype=float))
        emb_arr = np.stack(emb_list, axis=0)
        return (sample_ids, emb_arr)
    emb_arr = np.asarray(emb_obj, dtype=float)
    if emb_arr.ndim == 3:
        n = emb_arr.shape[0]
        sample_ids = [f'sample_{i:06d}' for i in range(n)]
        return (sample_ids, emb_arr)
    raise ValueError(f'Unsupported embedding format. Expected dict or ndarray [N, 4, D], got shape={emb_arr.shape}')

def build_sample_meta(sample_ids, risk_obj):
    rows = []
    if isinstance(risk_obj, dict):
        for sid in sample_ids:
            v = risk_obj.get(sid, None)
            (risk, event, time) = extract_meta_from_value(v)
            (pid, visit) = parse_sample_id(sid)
            rows.append({'sample_id': str(sid), 'pid': str(pid), 'visit': str(visit), 'risk': risk, 'event': event, 'time': time})
    else:
        arr = np.asarray(risk_obj, dtype=object)
        if len(arr) != len(sample_ids):
            raise ValueError(f'sample_risk length does not match embeddings: len(risk)={len(arr)}, len(embeddings)={len(sample_ids)}')
        for (sid, v) in zip(sample_ids, arr):
            (risk, event, time) = extract_meta_from_value(v)
            (pid, visit) = parse_sample_id(sid)
            rows.append({'sample_id': str(sid), 'pid': str(pid), 'visit': str(visit), 'risk': risk, 'event': event, 'time': time})
    meta = pd.DataFrame(rows)
    meta['pid'] = clean_id(meta['pid'])
    meta['visit'] = meta['visit'].astype(str)
    meta['risk'] = pd.to_numeric(meta['risk'], errors='coerce')
    meta['risk_norm'] = rank_norm(meta['risk'].values)
    meta['event'] = pd.to_numeric(meta['event'], errors='coerce')
    meta['event_binary'] = np.nan
    meta.loc[meta['event'] >= 0.5, 'event_binary'] = 1
    meta.loc[meta['event'] < 0.5, 'event_binary'] = 0
    meta['time'] = pd.to_numeric(meta['time'], errors='coerce')
    return meta

def load_age_meta(age_path):
    df_meta = pd.read_csv(str(OUTPUT_ROOT / age_path if (OUTPUT_ROOT / age_path).exists() else INPUT_ROOT / age_path), sep='\t', dtype={'ID': str}, low_memory=False)
    keep_cols = ['ID', 'age1992', 'sex']
    missing = [c for c in keep_cols if c not in df_meta.columns]
    if missing:
        raise ValueError(f'Missing columns in AGE_PATH: {missing}')
    df_meta = df_meta[keep_cols].copy()
    df_meta['ID'] = clean_id(df_meta['ID'])
    df_meta['age1992'] = pd.to_numeric(df_meta['age1992'], errors='coerce')
    df_meta['sex'] = pd.to_numeric(df_meta['sex'], errors='coerce')
    print('\nAGE meta before collapsing:')
    print('Rows:', len(df_meta))
    print('Unique IDs:', df_meta['ID'].nunique())
    df_meta = df_meta.sort_values(['ID']).groupby('ID', as_index=False).agg({'age1992': 'first', 'sex': 'first'})
    print('AGE meta after collapsing:')
    print('Rows:', len(df_meta))
    print('Unique IDs:', df_meta['ID'].nunique())
    return df_meta

def modality_normalize_embeddings(emb_arr):
    if emb_arr.ndim != 3:
        raise ValueError(f'Expected emb_arr [N, M, D], got {emb_arr.shape}')
    (n, m, d) = emb_arr.shape
    norm_arr = np.zeros_like(emb_arr, dtype=float)
    for mi in range(m):
        X = emb_arr[:, mi, :].astype(float)
        X[~np.isfinite(X)] = np.nan
        col_median = np.nanmedian(X, axis=0)
        bad_col = ~np.isfinite(col_median)
        col_median[bad_col] = 0.0
        inds = np.where(~np.isfinite(X))
        X[inds] = np.take(col_median, inds[1])
        scaler = StandardScaler()
        Xz = scaler.fit_transform(X)
        Xz[~np.isfinite(Xz)] = 0.0
        norm_arr[:, mi, :] = Xz
    return norm_arr

def diffusion_map_embedding(X, n_neighbors=30, n_components=6, diffusion_time=1, random_state=42):
    X = np.asarray(X, dtype=float)
    X[~np.isfinite(X)] = 0.0
    n = X.shape[0]
    if n < 5:
        raise ValueError('Too few points for diffusion map.')
    k = min(n_neighbors, n - 1)
    print(f'Building kNN graph: n={n}, k={k}')
    nn = NearestNeighbors(n_neighbors=k + 1, metric='euclidean', n_jobs=-1)
    nn.fit(X)
    (distances, indices) = nn.kneighbors(X)
    distances = distances[:, 1:]
    indices = indices[:, 1:]
    valid_dist = distances[np.isfinite(distances) & (distances > 0)]
    if len(valid_dist) == 0:
        sigma = 1.0
    else:
        sigma = np.median(valid_dist)
    if not np.isfinite(sigma) or sigma <= 1e-12:
        sigma = 1.0
    print(f'Diffusion kernel sigma = {sigma:.6g}')
    weights = np.exp(-distances ** 2 / (2 * sigma ** 2))
    weights[~np.isfinite(weights)] = 0.0
    row = np.repeat(np.arange(n), k)
    col = indices.reshape(-1)
    data = weights.reshape(-1)
    W = sparse.csr_matrix((data, (row, col)), shape=(n, n))
    W = W.maximum(W.T)
    d = np.asarray(W.sum(axis=1)).reshape(-1)
    d[d <= 1e-12] = 1e-12
    d_inv_sqrt = 1.0 / np.sqrt(d)
    D_inv_sqrt = sparse.diags(d_inv_sqrt)
    S = D_inv_sqrt @ W @ D_inv_sqrt
    n_eigs = min(n_components + 1, n - 1)
    print(f'Eigen decomposition: n_eigs={n_eigs}')
    (vals, vecs) = eigsh(S, k=n_eigs, which='LA')
    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    vals_nontrivial = vals[1:n_components + 1]
    vecs_nontrivial = vecs[:, 1:n_components + 1]
    psi = D_inv_sqrt @ vecs_nontrivial
    dc = psi * vals_nontrivial ** diffusion_time
    dc = np.asarray(dc, dtype=float)
    for i in range(dc.shape[1]):
        dc[:, i] = StandardScaler().fit_transform(dc[:, i].reshape(-1, 1)).reshape(-1)
    return (dc, vals)

def save_fig(fig, basename):
    pdf_path = build_output_path(f'{basename}.pdf')
    png_path = build_output_path(f'{basename}.png')
    fig.savefig(pdf_path, bbox_inches='tight', transparent=True)
    fig.savefig(png_path, dpi=DPI, bbox_inches='tight', transparent=True)
    print('Saved:', pdf_path)
    print('Saved:', png_path)

def basic_clean_ax(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

def plot_by_modality(point_df):
    (fig, ax) = plt.subplots(figsize=(6.2, 5.2))
    modalities = list(point_df['modality'].dropna().unique())
    cmap = plt.get_cmap('tab10')
    for (i, mod) in enumerate(modalities):
        sub = point_df[point_df['modality'] == mod]
        ax.scatter(sub['DC2'], sub['DC3'], s=POINT_SIZE, alpha=POINT_ALPHA, color=cmap(i % 10), linewidths=0, label=f'{mod}, n={len(sub)}')
    ax.set_title('Modality')
    basic_clean_ax(ax)
    ax.legend(frameon=False, markerscale=3, fontsize=9, loc='best')
    save_fig(fig, f'{PLOT_TAG}_modality')
    plt.close(fig)

def plot_by_risk(point_df):
    df = point_df.copy()
    risk = pd.to_numeric(df['risk_norm'], errors='coerce').values
    mask = np.isfinite(risk)
    (fig, ax) = plt.subplots(figsize=(6.2, 5.2))
    if mask.sum() > 0:
        (risk_clip, vmin, vmax) = clip_percentile(risk[mask], 2, 98)
        plot_values = np.full(len(df), np.nan)
        plot_values[mask] = np.clip(risk[mask], vmin, vmax)
        try:
            alpha_vals = density_alpha(df.loc[mask, 'DC2'].values, df.loc[mask, 'DC3'].values)
        except Exception:
            alpha_vals = POINT_ALPHA
        sc = ax.scatter(df.loc[mask, 'DC2'], df.loc[mask, 'DC3'], c=plot_values[mask], cmap='magma', s=POINT_SIZE, alpha=alpha_vals, vmin=vmin, vmax=vmax, linewidths=0)
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046)
        cbar.set_label('Normalized risk')
    if (~mask).sum() > 0:
        ax.scatter(df.loc[~mask, 'DC2'], df.loc[~mask, 'DC3'], s=POINT_SIZE, alpha=0.2, color='lightgray', linewidths=0)
    ax.set_title('Risk')
    basic_clean_ax(ax)
    save_fig(fig, f'{PLOT_TAG}_risk')
    plt.close(fig)

def plot_by_event(point_df):
    df = point_df.copy()
    event = pd.to_numeric(df['event_binary'], errors='coerce')
    mask0 = event == 0
    mask1 = event == 1
    missing = ~np.isfinite(event)
    (fig, ax) = plt.subplots(figsize=(6.2, 5.2))
    if missing.sum() > 0:
        ax.scatter(df.loc[missing, 'DC2'], df.loc[missing, 'DC3'], s=POINT_SIZE, alpha=0.2, color='lightgray', linewidths=0)
    if mask0.sum() > 0:
        ax.scatter(df.loc[mask0, 'DC2'], df.loc[mask0, 'DC3'], s=POINT_SIZE, alpha=0.45, color='#4C72B0', linewidths=0, label=f'No event, n={mask0.sum()}')
    if mask1.sum() > 0:
        ax.scatter(df.loc[mask1, 'DC2'], df.loc[mask1, 'DC3'], s=POINT_SIZE + 3, alpha=0.9, color='#C44E52', linewidths=0, label=f'Event, n={mask1.sum()}')
    ax.set_title('Event')
    basic_clean_ax(ax)
    legend_handles = [Line2D([0], [0], marker='o', color='w', label='No event', markerfacecolor='#4C72B0', markersize=6), Line2D([0], [0], marker='o', color='w', label='Event', markerfacecolor='#C44E52', markersize=6)]
    ax.legend(handles=legend_handles, frameon=False, loc='best')
    save_fig(fig, f'{PLOT_TAG}_event')
    plt.close(fig)

def plot_by_age(point_df):
    df = point_df.copy()
    age = pd.to_numeric(df['age'], errors='coerce').values
    mask = np.isfinite(age)
    (fig, ax) = plt.subplots(figsize=(6.2, 5.2))
    if mask.sum() > 0:
        try:
            alpha_vals = density_alpha(df.loc[mask, 'DC2'].values, df.loc[mask, 'DC3'].values)
        except Exception:
            alpha_vals = POINT_ALPHA
        sc = ax.scatter(df.loc[mask, 'DC2'], df.loc[mask, 'DC3'], c=age[mask], cmap='viridis', s=POINT_SIZE, alpha=alpha_vals, linewidths=0)
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046)
        cbar.set_label('Age')
    if (~mask).sum() > 0:
        ax.scatter(df.loc[~mask, 'DC2'], df.loc[~mask, 'DC3'], s=POINT_SIZE, alpha=0.2, color='lightgray', linewidths=0)
    ax.set_title('Age')
    basic_clean_ax(ax)
    save_fig(fig, f'{PLOT_TAG}_age')
    plt.close(fig)

def plot_fourpanel(point_df):
    df = point_df.copy()
    (fig, axes) = plt.subplots(2, 2, figsize=(10, 8))
    ax_mod = axes[0, 0]
    ax_risk = axes[0, 1]
    ax_event = axes[1, 0]
    ax_age = axes[1, 1]
    modalities = list(df['modality'].dropna().unique())
    cmap_mod = plt.get_cmap('tab10')
    for (i, mod) in enumerate(modalities):
        sub = df[df['modality'] == mod]
        ax_mod.scatter(sub['DC2'], sub['DC3'], s=POINT_SIZE, alpha=POINT_ALPHA, color=cmap_mod(i % 10), linewidths=0, label=mod)
    ax_mod.set_title('Modality')
    ax_mod.legend(frameon=False, markerscale=3, fontsize=8, loc='best')
    risk = pd.to_numeric(df['risk_norm'], errors='coerce').values
    mask_risk = np.isfinite(risk)
    if mask_risk.sum() > 0:
        (_, vmin, vmax) = clip_percentile(risk[mask_risk], 2, 98)
        try:
            alpha_vals = density_alpha(df.loc[mask_risk, 'DC2'].values, df.loc[mask_risk, 'DC3'].values)
        except Exception:
            alpha_vals = POINT_ALPHA
        sc1 = ax_risk.scatter(df.loc[mask_risk, 'DC2'], df.loc[mask_risk, 'DC3'], c=np.clip(risk[mask_risk], vmin, vmax), cmap='magma', s=POINT_SIZE, alpha=alpha_vals, vmin=vmin, vmax=vmax, linewidths=0)
        fig.colorbar(sc1, ax=ax_risk, fraction=0.046)
    ax_risk.set_title('Risk')
    event = pd.to_numeric(df['event_binary'], errors='coerce')
    mask0 = event == 0
    mask1 = event == 1
    if mask0.sum() > 0:
        ax_event.scatter(df.loc[mask0, 'DC2'], df.loc[mask0, 'DC3'], s=POINT_SIZE, alpha=0.45, color='#4C72B0', linewidths=0, label='No event')
    if mask1.sum() > 0:
        ax_event.scatter(df.loc[mask1, 'DC2'], df.loc[mask1, 'DC3'], s=POINT_SIZE + 3, alpha=0.9, color='#C44E52', linewidths=0, label='Event')
    ax_event.set_title('Event')
    ax_event.legend(frameon=False, markerscale=3, fontsize=8, loc='best')
    age = pd.to_numeric(df['age'], errors='coerce').values
    mask_age = np.isfinite(age)
    if mask_age.sum() > 0:
        try:
            alpha_vals = density_alpha(df.loc[mask_age, 'DC2'].values, df.loc[mask_age, 'DC3'].values)
        except Exception:
            alpha_vals = POINT_ALPHA
        sc3 = ax_age.scatter(df.loc[mask_age, 'DC3'], df.loc[mask_age, 'DC3'], c=age[mask_age], cmap='viridis', s=POINT_SIZE, alpha=alpha_vals, linewidths=0)
        fig.colorbar(sc3, ax=ax_age, fraction=0.046)
    ax_age.set_title('Age')
    for ax in [ax_mod, ax_risk, ax_event, ax_age]:
        basic_clean_ax(ax)
    out_pdf = build_output_path(f'{PLOT_TAG}_fourpanel.pdf')
    out_png = build_output_path(f'{PLOT_TAG}_fourpanel.png')
    plt.tight_layout()
    fig.savefig(out_pdf, dpi=DPI, bbox_inches='tight', transparent=True)
    fig.savefig(out_png, dpi=DPI, bbox_inches='tight', transparent=True)
    print('Saved:', out_pdf)
    print('Saved:', out_png)
    plt.close(fig)

def main():
    print('===================================================')
    print('Loading files')
    print('===================================================')
    print('Embedding:', EMB_PATH)
    print('Attention:', ATTN_PATH)
    print('Risk:', RISK_PATH)
    print('Age meta:', AGE_PATH)
    emb_obj = load_npy_any(EMB_PATH)
    risk_obj = load_npy_any(RISK_PATH)
    if ATTN_PATH.exists():
        attn_obj = load_npy_any(ATTN_PATH)
        print('Attention file loaded but NOT used for fusion.')
        if isinstance(attn_obj, dict):
            print('Attention object type: dict, n =', len(attn_obj))
        else:
            print('Attention shape:', np.asarray(attn_obj, dtype=object).shape)
    (sample_ids, emb_arr) = infer_sample_order_from_embedding(emb_obj)
    print('\nRaw embedding shape:', emb_arr.shape)
    print('Number of samples:', len(sample_ids))
    print('First 5 sample IDs:', sample_ids[:5])
    if emb_arr.ndim != 3:
        raise ValueError(f'Expected embedding shape [N, M, D], got {emb_arr.shape}')
    (n_sample, n_modality, dim) = emb_arr.shape
    if n_modality != len(MODALITY_NAMES):
        print(f'WARNING: embedding has {n_modality} modalities, but MODALITY_NAMES has {len(MODALITY_NAMES)} names.')
        mod_names = [f'Modality_{i}' for i in range(n_modality)]
    else:
        mod_names = MODALITY_NAMES
    print('Modality names:', mod_names)
    sample_meta = build_sample_meta(sample_ids, risk_obj)
    age_meta = load_age_meta(AGE_PATH)
    sample_meta = sample_meta.merge(age_meta, left_on='pid', right_on='ID', how='left')
    sample_meta['age'] = np.nan
    sample_meta.loc[sample_meta['visit'].astype(str) == '12', 'age'] = sample_meta.loc[sample_meta['visit'].astype(str) == '12', 'age1992'] + 20
    if len(sample_meta) != emb_arr.shape[0]:
        raise ValueError(f'sample_meta and emb_arr length mismatch before visit filtering: len(sample_meta)={len(sample_meta)}, emb_arr.shape[0]={emb_arr.shape[0]}. This usually means AGE_PATH merge expanded rows due to duplicated ID.')
    visit_mask = (sample_meta['visit'].astype(str) == TARGET_VISIT).values
    print('\n===================================================')
    print(f'Filtering visit == {TARGET_VISIT}')
    print('===================================================')
    print('Before filtering samples:', len(sample_meta))
    print('Embedding samples before filtering:', emb_arr.shape[0])
    print('Visit counts before filtering:')
    print(sample_meta['visit'].value_counts(dropna=False).sort_index())
    sample_meta = sample_meta.loc[visit_mask].copy()
    emb_arr = emb_arr[visit_mask, :, :]
    print('After filtering samples:', len(sample_meta))
    print('Embedding samples after filtering:', emb_arr.shape[0])
    print('After filtering samples:', len(sample_meta))
    if len(sample_meta) == 0:
        raise ValueError(f'No samples left after filtering visit == {TARGET_VISIT}. Please check sample_id format, e.g. whether it is pid-12.')
    sample_meta['risk'] = pd.to_numeric(sample_meta['risk'], errors='coerce')
    sample_meta['risk_norm'] = rank_norm(sample_meta['risk'].values)
    sample_meta['event_binary'] = pd.to_numeric(sample_meta['event_binary'], errors='coerce')
    sample_meta['age'] = pd.to_numeric(sample_meta['age'], errors='coerce')
    sample_meta['sex'] = pd.to_numeric(sample_meta['sex'], errors='coerce')
    print('\n===================================================')
    print('Sample-level metadata summary after visit12 filtering')
    print('===================================================')
    print(sample_meta.head())
    print('\nN visit12 samples:', len(sample_meta))
    print('Risk finite:', np.isfinite(sample_meta['risk']).sum())
    print('Risk_norm finite:', np.isfinite(sample_meta['risk_norm']).sum())
    print('Event finite:', np.isfinite(sample_meta['event_binary']).sum())
    print('Age finite:', np.isfinite(sample_meta['age']).sum())
    print('Sex finite:', np.isfinite(sample_meta['sex']).sum())
    if np.isfinite(sample_meta['event_binary']).sum() > 0:
        print('\nEvent count:')
        print(sample_meta['event_binary'].value_counts(dropna=False).sort_index())
    meta_path = build_output_path(f'{PLOT_TAG}_sample_meta_visit12.csv')
    sample_meta.to_csv(meta_path, index=False)
    print('\nSaved sample meta:', meta_path)
    print('\n===================================================')
    print('Normalize embeddings within each modality')
    print('===================================================')
    emb_norm = modality_normalize_embeddings(emb_arr)
    rows = []
    X_list = []
    sample_meta = sample_meta.reset_index(drop=True)
    for i in range(len(sample_meta)):
        sid = sample_meta.loc[i, 'sample_id']
        for mi in range(n_modality):
            X_list.append(emb_norm[i, mi, :])
            rows.append({'sample_id': sid, 'pid': sample_meta.loc[i, 'pid'], 'visit': sample_meta.loc[i, 'visit'], 'modality_idx': mi, 'modality': mod_names[mi], 'risk': sample_meta.loc[i, 'risk'], 'risk_norm': sample_meta.loc[i, 'risk_norm'], 'event': sample_meta.loc[i, 'event'], 'event_binary': sample_meta.loc[i, 'event_binary'], 'time': sample_meta.loc[i, 'time'], 'age': sample_meta.loc[i, 'age'], 'sex': sample_meta.loc[i, 'sex']})
    X_points = np.vstack(X_list)
    point_df = pd.DataFrame(rows)
    print('\n===================================================')
    print('Stacked modality-point data')
    print('===================================================')
    print('Stacked point feature shape:', X_points.shape)
    print('Point dataframe shape:', point_df.shape)
    print('Expected points = N visit12 samples x M modalities:', len(sample_meta) * n_modality)
    print(point_df['modality'].value_counts())
    print('\n===================================================')
    print('Running Diffusion Map')
    print('===================================================')
    (dc, eigvals) = diffusion_map_embedding(X_points, n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS, diffusion_time=DIFFUSION_TIME, random_state=RANDOM_STATE)
    for i in range(dc.shape[1]):
        point_df[f'DC{i + 1}'] = dc[:, i]
    print('Diffusion components shape:', dc.shape)
    print('Top eigenvalues:')
    print(eigvals[:min(10, len(eigvals))])
    point_path = build_output_path(f'{PLOT_TAG}_point_level_embeddings_visit12.csv')
    point_df.to_csv(point_path, index=False)
    print('\nSaved point-level dataframe:', point_path)
    print('\n===================================================')
    print('Plotting 4 separate figures')
    print('===================================================')
    plot_by_modality(point_df)
    plot_by_risk(point_df)
    plot_by_event(point_df)
    plot_by_age(point_df)
    if SAVE_FOURPANEL:
        print('\n===================================================')
        print('Plotting optional 2x2 fourpanel')
        print('===================================================')
        plot_fourpanel(point_df)
    print('\nDone.')
if __name__ == '__main__':
    main()


### Supporting analysis for Figure 4f — Modality and fused diffusion maps


In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import eigsh
from scipy.stats import rankdata, spearmanr, pearsonr
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE_DIR = Path('fig3/code_save_6_7')
OUT_DIR = Path('fig4_final/embedding_v3')
OUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_PATH = BASE_DIR / 'sample_embeddings_ICVD_test.npy'
ATTN_PATH = BASE_DIR / 'sample_attn_ICVD_test.npy'
RISK_PATH = BASE_DIR / 'sample_risk_ICVD_test.npy'
AGE_PATH = 'data/private/data_2klh_longitudinal.tsv'
TARGET_VISIT = '12'
MODALITY_NAMES = ['Fundus', 'SNP', 'Soma', 'ECG']
N_NEIGHBORS = 30
N_COMPONENTS = 10
DIFFUSION_TIME = 1
POINT_SIZE = 6
POINT_ALPHA = 0.75
DPI = 320
PLOT_TAG = 'ICVD_visit12_attn_fused_diffmap'

def build_output_path(filename: str, out_dir: Path=OUT_DIR) -> str:
    out_dir.mkdir(parents=True, exist_ok=True)
    (stem, ext) = os.path.splitext(filename)
    candidate = out_dir / filename
    if not candidate.exists():
        return str(candidate)
    i = 1
    while True:
        candidate = out_dir / f'{stem}_v{i:02d}{ext}'
        if not candidate.exists():
            return str(candidate)
        i += 1

def load_npy_any(path: Path):
    obj = np.load(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path), allow_pickle=True)
    if isinstance(obj, np.ndarray) and obj.shape == () and (obj.dtype == object):
        return obj.item()
    return obj

def clean_id(x):
    x = pd.Series(x).astype(str)
    x = x.str.replace('.0', '', regex=False)
    x = x.str.strip()
    return x

def to_float_or_nan(x):
    try:
        if x is None:
            return np.nan
        if isinstance(x, (list, tuple, np.ndarray)):
            arr = np.array(x, dtype=object).reshape(-1)
            if len(arr) == 1:
                x = arr[0]
            else:
                return np.nan
        return float(x)
    except Exception:
        return np.nan

def parse_sample_id(sample_key: str):
    s = str(sample_key)
    if '-' in s:
        (pid, visit) = s.split('-', 1)
        return (str(pid), str(visit))
    return (str(s), 'UNK')

def parse_risk_event_time(v):
    risk = np.nan
    event = np.nan
    time = np.nan
    if isinstance(v, dict):
        for k in ['risk', 'pred', 'score', 'rank', 'yhat', 'risk_raw', 'risk_norm', 'prob']:
            if k in v:
                risk = to_float_or_nan(v.get(k))
                if np.isfinite(risk):
                    break
        if not np.isfinite(risk):
            for val in v.values():
                num = to_float_or_nan(val)
                if np.isfinite(num):
                    risk = num
                    break
        for k in ['event', 'event_binary', 'label', 'y', 'outcome', 'case', 'status']:
            if k in v:
                event = to_float_or_nan(v.get(k))
                if np.isfinite(event):
                    break
        for k in ['time', 'tte', 'time_to_event', 'surv_time', 'followup_time', 'duration']:
            if k in v:
                time = to_float_or_nan(v.get(k))
                if np.isfinite(time):
                    break
    elif isinstance(v, (list, tuple, np.ndarray)):
        arr = np.array(v, dtype=object).reshape(-1)
        if len(arr) > 0:
            risk = to_float_or_nan(arr[0])
        if len(arr) > 1:
            event = to_float_or_nan(arr[1])
        if len(arr) > 2:
            time = to_float_or_nan(arr[2])
    else:
        risk = to_float_or_nan(v)
    return (risk, event, time)

def rank_norm(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)
    out = np.full(len(x), np.nan, dtype=float)
    if ok.sum() == 0:
        return out
    r = rankdata(x[ok], method='average')
    if len(r) == 1:
        out[ok] = 0.5
    else:
        out[ok] = (r - 1) / (len(r) - 1)
    return out

def fuse_attn_embedding(emb, attn):
    E = np.asarray(emb, dtype=float)
    A = np.asarray(attn, dtype=float).reshape(-1)
    if E.ndim == 1:
        return E.astype(float)
    if E.ndim != 2:
        raise ValueError(f'Unexpected embedding shape: {E.shape}')
    n_mod = E.shape[0]
    if A.shape[0] != n_mod:
        A = np.ones(n_mod, dtype=float) / max(n_mod, 1)
    else:
        s = A.sum()
        if not np.isfinite(s) or abs(s) < 1e-12:
            A = np.ones(n_mod, dtype=float) / max(n_mod, 1)
        else:
            A = A / s
    return (A[:, None] * E).sum(axis=0)

def load_age_meta(age_path):
    df_meta = pd.read_csv(str(OUTPUT_ROOT / age_path if (OUTPUT_ROOT / age_path).exists() else INPUT_ROOT / age_path), sep='\t', dtype={'ID': str}, low_memory=False)
    keep_cols = ['ID', 'age1992', 'sex']
    missing = [c for c in keep_cols if c not in df_meta.columns]
    if missing:
        raise ValueError(f'Missing columns in AGE_PATH: {missing}')
    df_meta = df_meta[keep_cols].copy()
    df_meta['ID'] = clean_id(df_meta['ID'])
    df_meta['age1992'] = pd.to_numeric(df_meta['age1992'], errors='coerce')
    df_meta['sex'] = pd.to_numeric(df_meta['sex'], errors='coerce')
    print('\nAGE meta before collapsing:')
    print('Rows:', len(df_meta))
    print('Unique IDs:', df_meta['ID'].nunique())
    df_meta = df_meta.sort_values('ID').groupby('ID', as_index=False).agg({'age1992': 'first', 'sex': 'first'})
    print('AGE meta after collapsing:')
    print('Rows:', len(df_meta))
    print('Unique IDs:', df_meta['ID'].nunique())
    return df_meta

def standardize_matrix(X):
    X = np.asarray(X, dtype=float)
    X[~np.isfinite(X)] = np.nan
    col_median = np.nanmedian(X, axis=0)
    bad_col = ~np.isfinite(col_median)
    col_median[bad_col] = 0.0
    inds = np.where(~np.isfinite(X))
    X[inds] = np.take(col_median, inds[1])
    scaler = StandardScaler()
    Xz = scaler.fit_transform(X)
    Xz[~np.isfinite(Xz)] = 0.0
    return Xz

def diffusion_map_embedding(X, n_neighbors=30, n_components=10, diffusion_time=1):
    X = np.asarray(X, dtype=float)
    X[~np.isfinite(X)] = 0.0
    n = X.shape[0]
    if n < 5:
        raise ValueError('Too few points for diffusion map.')
    k = min(n_neighbors, n - 1)
    print(f'Building kNN graph: n={n}, k={k}')
    nn = NearestNeighbors(n_neighbors=k + 1, metric='euclidean', n_jobs=-1)
    nn.fit(X)
    (distances, indices) = nn.kneighbors(X)
    distances = distances[:, 1:]
    indices = indices[:, 1:]
    valid_dist = distances[np.isfinite(distances) & (distances > 0)]
    if len(valid_dist) == 0:
        sigma = 1.0
    else:
        sigma = np.median(valid_dist)
    if not np.isfinite(sigma) or sigma <= 1e-12:
        sigma = 1.0
    print(f'Diffusion sigma = {sigma:.6g}')
    weights = np.exp(-distances ** 2 / (2 * sigma ** 2))
    weights[~np.isfinite(weights)] = 0.0
    row = np.repeat(np.arange(n), k)
    col = indices.reshape(-1)
    data = weights.reshape(-1)
    W = sparse.csr_matrix((data, (row, col)), shape=(n, n))
    W = W.maximum(W.T)
    d = np.asarray(W.sum(axis=1)).reshape(-1)
    d[d <= 1e-12] = 1e-12
    d_inv_sqrt = 1.0 / np.sqrt(d)
    D_inv_sqrt = sparse.diags(d_inv_sqrt)
    S = D_inv_sqrt @ W @ D_inv_sqrt
    n_eigs = min(n_components + 1, n - 1)
    print(f'Eigen decomposition: n_eigs={n_eigs}')
    (vals, vecs) = eigsh(S, k=n_eigs, which='LA')
    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    vals_nontrivial = vals[1:n_components + 1]
    vecs_nontrivial = vecs[:, 1:n_components + 1]
    psi = D_inv_sqrt @ vecs_nontrivial
    dc = psi * vals_nontrivial ** diffusion_time
    dc = np.asarray(dc, dtype=float)
    for i in range(dc.shape[1]):
        dc[:, i] = StandardScaler().fit_transform(dc[:, i].reshape(-1, 1)).reshape(-1)
    return (dc, vals)

def compute_dc_risk_corr(df, risk_col='risk_norm', dc_prefix='DC'):
    dc_cols = [c for c in df.columns if c.startswith(dc_prefix) and c[len(dc_prefix):].isdigit()]
    dc_cols = sorted(dc_cols, key=lambda x: int(x.replace(dc_prefix, '')))
    results = []
    for dc in dc_cols:
        x = pd.to_numeric(df[dc], errors='coerce').values
        y = pd.to_numeric(df[risk_col], errors='coerce').values
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() >= 5:
            (pr, pp) = pearsonr(x[mask], y[mask])
            (sr, sp) = spearmanr(x[mask], y[mask])
        else:
            (pr, pp, sr, sp) = (np.nan, np.nan, np.nan, np.nan)
        results.append({'dc': dc, 'dc_num': int(dc.replace(dc_prefix, '')), 'n': int(mask.sum()), 'pearson_r': pr, 'pearson_p': pp, 'spearman_r': sr, 'spearman_p': sp, 'abs_pearson_r': abs(pr) if np.isfinite(pr) else np.nan, 'abs_spearman_r': abs(sr) if np.isfinite(sr) else np.nan})
    res = pd.DataFrame(results).sort_values('dc_num')
    return res

def pick_top2_dc(corr_df, metric='abs_spearman_r'):
    sub = corr_df.sort_values(metric, ascending=False).reset_index(drop=True)
    if len(sub) == 0:
        raise ValueError('No DC correlation results found.')
    if len(sub) == 1:
        return (sub.loc[0, 'dc'], sub.loc[0, 'dc'])
    return (sub.loc[0, 'dc'], sub.loc[1, 'dc'])

def add_dc_columns(meta_df, dc):
    out = meta_df.copy().reset_index(drop=True)
    for i in range(dc.shape[1]):
        out[f'DC{i + 1}'] = dc[:, i]
    return out

def save_fig(fig, basename):
    pdf_path = build_output_path(f'{basename}.pdf')
    png_path = build_output_path(f'{basename}.png')
    fig.savefig(pdf_path, bbox_inches='tight', transparent=True)
    fig.savefig(png_path, dpi=DPI, bbox_inches='tight', transparent=True)
    print('Saved:', pdf_path)
    print('Saved:', png_path)

def basic_clean_ax(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

def load_visit12_dataset():
    print('===================================================')
    print('Loading core files')
    print('===================================================')
    print('Embedding:', EMB_PATH)
    print('Attention:', ATTN_PATH)
    print('Risk:', RISK_PATH)
    print('Age meta:', AGE_PATH)
    emb_obj = load_npy_any(EMB_PATH)
    attn_obj = load_npy_any(ATTN_PATH)
    risk_obj = load_npy_any(RISK_PATH)
    if not isinstance(emb_obj, dict):
        raise ValueError('Embedding npy is expected to be dict.')
    if not isinstance(attn_obj, dict):
        raise ValueError('Attention npy is expected to be dict.')
    if not isinstance(risk_obj, dict):
        raise ValueError('Risk npy is expected to be dict.')
    common_keys = sorted(set(emb_obj.keys()) & set(attn_obj.keys()) & set(risk_obj.keys()))
    print('Common aligned keys:', len(common_keys))
    rows = []
    emb_list = []
    attn_list = []
    for k in common_keys:
        emb = np.asarray(emb_obj[k], dtype=float)
        attn = np.asarray(attn_obj[k], dtype=float).reshape(-1)
        if emb.ndim != 2:
            continue
        if emb.shape[0] != len(MODALITY_NAMES):
            continue
        (risk_raw, event_raw, time_raw) = parse_risk_event_time(risk_obj[k])
        (pid, visit) = parse_sample_id(k)
        rows.append({'sample_key': str(k), 'pid': str(pid), 'visit': str(visit), 'risk_raw': risk_raw, 'event_raw': event_raw, 'time_raw': time_raw})
        emb_list.append(emb)
        attn_list.append(attn)
    if len(rows) == 0:
        raise ValueError('No valid aligned samples found.')
    meta = pd.DataFrame(rows)
    emb_arr = np.stack(emb_list, axis=0)
    attn_arr = np.stack(attn_list, axis=0)
    print('\nRaw loaded samples:', len(meta))
    print('Embedding array shape:', emb_arr.shape)
    print('Attention array shape:', attn_arr.shape)
    print('Visit counts before filtering:')
    print(meta['visit'].value_counts(dropna=False).sort_index())
    meta['pid'] = clean_id(meta['pid'])
    meta['visit'] = meta['visit'].astype(str)
    meta['risk_raw'] = pd.to_numeric(meta['risk_raw'], errors='coerce')
    meta['event'] = pd.to_numeric(meta['event_raw'], errors='coerce')
    meta['time'] = pd.to_numeric(meta['time_raw'], errors='coerce')
    meta['event_binary'] = np.nan
    meta.loc[meta['event'] >= 0.5, 'event_binary'] = 1
    meta.loc[meta['event'] < 0.5, 'event_binary'] = 0
    age_meta = load_age_meta(AGE_PATH)
    meta = meta.merge(age_meta, left_on='pid', right_on='ID', how='left')
    if len(meta) != emb_arr.shape[0]:
        raise ValueError(f'Length mismatch after age merge: len(meta)={len(meta)}, emb_arr={emb_arr.shape[0]}')
    meta['age'] = np.nan
    meta.loc[meta['visit'] == '12', 'age'] = meta.loc[meta['visit'] == '12', 'age1992'] + 20
    meta.loc[meta['visit'] == '20', 'age'] = meta.loc[meta['visit'] == '20', 'age1992'] + 28
    visit_mask = (meta['visit'] == TARGET_VISIT).values
    meta = meta.loc[visit_mask].copy().reset_index(drop=True)
    emb_arr = emb_arr[visit_mask, :, :]
    attn_arr = attn_arr[visit_mask, :]
    if len(meta) == 0:
        raise ValueError('No visit12 samples left after filtering.')
    meta['risk_norm'] = rank_norm(meta['risk_raw'].values)
    meta['age'] = pd.to_numeric(meta['age'], errors='coerce')
    meta['sex'] = pd.to_numeric(meta['sex'], errors='coerce')
    print('\n===================================================')
    print('After visit12 filtering')
    print('===================================================')
    print('N samples:', len(meta))
    print('Embedding shape:', emb_arr.shape)
    print('Attention shape:', attn_arr.shape)
    print('Risk finite:', np.isfinite(meta['risk_norm']).sum())
    print('Age finite:', np.isfinite(meta['age']).sum())
    print('Event finite:', np.isfinite(meta['event_binary']).sum())
    print('\nEvent count:')
    print(meta['event_binary'].value_counts(dropna=False).sort_index())
    meta_path = build_output_path(f'{PLOT_TAG}_sample_meta_visit12.csv')
    meta.to_csv(meta_path, index=False)
    print('Saved:', meta_path)
    return (meta, emb_arr, attn_arr)

def run_modality_diffmaps(meta, emb_arr):
    modality_results = {}
    n_mod = emb_arr.shape[1]
    if n_mod != len(MODALITY_NAMES):
        raise ValueError(f'Embedding modality dim = {n_mod}, but MODALITY_NAMES has {len(MODALITY_NAMES)} names.')
    for (m, mod_name) in enumerate(MODALITY_NAMES):
        print('\n===================================================')
        print(f'Running modality DiffMap: {mod_name}')
        print('===================================================')
        X = emb_arr[:, m, :]
        X = standardize_matrix(X)
        (dc, eigvals) = diffusion_map_embedding(X, n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS, diffusion_time=DIFFUSION_TIME)
        df_dc = add_dc_columns(meta, dc)
        corr_df = compute_dc_risk_corr(df_dc, risk_col='risk_norm', dc_prefix='DC')
        (xdc, ydc) = pick_top2_dc(corr_df, metric='abs_spearman_r')
        corr_path = build_output_path(f'{PLOT_TAG}_{mod_name}_DC_risk_corr.csv')
        coords_path = build_output_path(f'{PLOT_TAG}_{mod_name}_diffmap_coords.csv')
        corr_df.to_csv(corr_path, index=False)
        df_dc.to_csv(coords_path, index=False)
        print('Best two DCs for', mod_name, ':', xdc, ydc)
        print(corr_df.sort_values('abs_spearman_r', ascending=False).head(5))
        print('Saved:', corr_path)
        print('Saved:', coords_path)
        modality_results[mod_name] = {'X': X, 'dc': dc, 'eigvals': eigvals, 'df': df_dc, 'corr': corr_df, 'xdc': xdc, 'ydc': ydc}
    return modality_results

def plot_modality_fourpanel(modality_results):
    (fig, axes) = plt.subplots(2, 2, figsize=(10.5, 8.5))
    axes = axes.flatten()
    order = ['Fundus', 'SNP', 'Soma', 'ECG']
    all_risk = []
    for mod in order:
        df = modality_results[mod]['df']
        all_risk.extend(pd.to_numeric(df['risk_norm'], errors='coerce').values.tolist())
    all_risk = np.array(all_risk, dtype=float)
    valid_risk = all_risk[np.isfinite(all_risk)]
    if len(valid_risk) > 0:
        (vmin, vmax) = np.nanpercentile(valid_risk, [2, 98])
    else:
        (vmin, vmax) = (0.0, 1.0)
    sc = None
    for (ax, mod) in zip(axes, order):
        obj = modality_results[mod]
        df = obj['df']
        corr_df = obj['corr']
        xdc = obj['xdc']
        ydc = obj['ydc']
        sub = corr_df.set_index('dc')
        rx = sub.loc[xdc, 'spearman_r'] if xdc in sub.index else np.nan
        ry = sub.loc[ydc, 'spearman_r'] if ydc in sub.index else np.nan
        x = pd.to_numeric(df[xdc], errors='coerce').values
        y = pd.to_numeric(df[ydc], errors='coerce').values
        c = pd.to_numeric(df['risk_norm'], errors='coerce').values
        mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(c)
        sc = ax.scatter(x[mask], y[mask], c=np.clip(c[mask], vmin, vmax), cmap='magma', s=POINT_SIZE, alpha=POINT_ALPHA, vmin=vmin, vmax=vmax, linewidths=0)
        ax.set_title(f'{mod}\n{xdc} (ρ={rx:.3f}) vs {ydc} (ρ={ry:.3f})', fontsize=11)
        basic_clean_ax(ax)
    cbar = fig.colorbar(sc, ax=axes.tolist(), fraction=0.03, pad=0.02)
    cbar.set_label('Normalized risk')
    plt.tight_layout()
    save_fig(fig, f'{PLOT_TAG}_modalities_fourpanel')
    plt.close(fig)

def run_fused_diffmap(meta, emb_arr, attn_arr):
    print('\n===================================================')
    print('Running fused attention-weighted DiffMap')
    print('===================================================')
    fused_list = []
    for i in range(len(meta)):
        fused = fuse_attn_embedding(emb_arr[i], attn_arr[i])
        fused_list.append(fused)
    X_fused = np.vstack(fused_list).astype(float)
    X_fused = standardize_matrix(X_fused)
    (dc, eigvals) = diffusion_map_embedding(X_fused, n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS, diffusion_time=DIFFUSION_TIME)
    df_dc = add_dc_columns(meta, dc)
    corr_df = compute_dc_risk_corr(df_dc, risk_col='risk_norm', dc_prefix='DC')
    (xdc, ydc) = pick_top2_dc(corr_df, metric='abs_spearman_r')
    corr_path = build_output_path(f'{PLOT_TAG}_FUSED_DC_risk_corr.csv')
    coords_path = build_output_path(f'{PLOT_TAG}_FUSED_diffmap_coords.csv')
    corr_df.to_csv(corr_path, index=False)
    df_dc.to_csv(coords_path, index=False)
    print('Best two DCs for FUSED:', xdc, ydc)
    print(corr_df.sort_values('abs_spearman_r', ascending=False).head(5))
    print('Saved:', corr_path)
    print('Saved:', coords_path)
    return {'X': X_fused, 'dc': dc, 'eigvals': eigvals, 'df': df_dc, 'corr': corr_df, 'xdc': xdc, 'ydc': ydc}

def plot_fused_fourpanel(fused_result):
    df = fused_result['df']
    xdc = fused_result['xdc']
    ydc = fused_result['ydc']
    corr_df = fused_result['corr']
    sub = corr_df.set_index('dc')
    rx = sub.loc[xdc, 'spearman_r'] if xdc in sub.index else np.nan
    ry = sub.loc[ydc, 'spearman_r'] if ydc in sub.index else np.nan
    x = pd.to_numeric(df[xdc], errors='coerce').values
    y = pd.to_numeric(df[ydc], errors='coerce').values
    risk = pd.to_numeric(df['risk_norm'], errors='coerce').values
    age = pd.to_numeric(df['age'], errors='coerce').values
    event = pd.to_numeric(df['event_binary'], errors='coerce').values
    (fig, axes) = plt.subplots(2, 2, figsize=(10.5, 8.5))
    ax_plain = axes[0, 0]
    ax_risk = axes[0, 1]
    ax_age = axes[1, 0]
    ax_event = axes[1, 1]
    mask_plain = np.isfinite(x) & np.isfinite(y)
    ax_plain.scatter(x[mask_plain], y[mask_plain], s=POINT_SIZE, alpha=0.55, color='gray', linewidths=0)
    ax_plain.set_title(f'Fused DiffMap\n{xdc} (ρ={rx:.3f}) vs {ydc} (ρ={ry:.3f})')
    basic_clean_ax(ax_plain)
    mask_risk = np.isfinite(x) & np.isfinite(y) & np.isfinite(risk)
    if mask_risk.sum() > 0:
        (vmin_r, vmax_r) = np.nanpercentile(risk[mask_risk], [2, 98])
        sc1 = ax_risk.scatter(x[mask_risk], y[mask_risk], c=np.clip(risk[mask_risk], vmin_r, vmax_r), cmap='magma', s=POINT_SIZE, alpha=POINT_ALPHA, vmin=vmin_r, vmax=vmax_r, linewidths=0)
        fig.colorbar(sc1, ax=ax_risk, fraction=0.046).set_label('Normalized risk')
    ax_risk.set_title('Risk')
    basic_clean_ax(ax_risk)
    mask_age = np.isfinite(x) & np.isfinite(y) & np.isfinite(age)
    if mask_age.sum() > 0:
        sc2 = ax_age.scatter(x[mask_age], y[mask_age], c=age[mask_age], cmap='viridis', s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0)
        fig.colorbar(sc2, ax=ax_age, fraction=0.046).set_label('Age')
    ax_age.set_title('Age')
    basic_clean_ax(ax_age)
    mask0 = np.isfinite(x) & np.isfinite(y) & (event == 0)
    mask1 = np.isfinite(x) & np.isfinite(y) & (event == 1)
    maskm = np.isfinite(x) & np.isfinite(y) & ~np.isfinite(event)
    if maskm.sum() > 0:
        ax_event.scatter(x[maskm], y[maskm], s=POINT_SIZE, alpha=0.2, color='lightgray', linewidths=0)
    if mask0.sum() > 0:
        ax_event.scatter(x[mask0], y[mask0], s=POINT_SIZE, alpha=0.5, color='#4C72B0', linewidths=0)
    if mask1.sum() > 0:
        ax_event.scatter(x[mask1], y[mask1], s=POINT_SIZE + 2, alpha=0.9, color='#C44E52', linewidths=0)
    ax_event.set_title('Event')
    basic_clean_ax(ax_event)
    event_legend = [Line2D([0], [0], marker='o', color='w', label='No event', markerfacecolor='#4C72B0', markersize=6), Line2D([0], [0], marker='o', color='w', label='Event', markerfacecolor='#C44E52', markersize=6)]
    ax_event.legend(handles=event_legend, frameon=False, loc='best')
    plt.tight_layout()
    save_fig(fig, f'{PLOT_TAG}_FUSED_fourpanel_plain_risk_age_event')
    plt.close(fig)

def main():
    (meta, emb_arr, attn_arr) = load_visit12_dataset()
    modality_results = run_modality_diffmaps(meta, emb_arr)
    plot_modality_fourpanel(modality_results)
    fused_result = run_fused_diffmap(meta, emb_arr, attn_arr)
    plot_fused_fourpanel(fused_result)
    print('\n===================================================')
    print('Done.')
    print('All outputs saved to:')
    print(OUT_DIR)
    print('===================================================')
if __name__ == '__main__':
    main()


### Supporting analysis for Figure 4f — PCA and diffusion-map views


In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import eigsh
from scipy.stats import rankdata, spearmanr, pearsonr
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE_DIR = Path('fig3/code_save_6_7')
OUT_DIR = Path('fig4_final/embeddingv4')
OUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_PATH = BASE_DIR / 'sample_embeddings_ICVD_test.npy'
ATTN_PATH = BASE_DIR / 'sample_attn_ICVD_test.npy'
RISK_PATH = BASE_DIR / 'sample_risk_ICVD_test.npy'
AGE_PATH = 'data/private/data_2klh_longitudinal.tsv'
TARGET_VISIT = '12'
MODALITY_NAMES = ['Fundus', 'SNP', 'Soma', 'ECG']
N_COMPONENTS = 15
N_NEIGHBORS = 30
DIFFUSION_TIME = 1
TOP_AXIS_METRIC = 'abs_pearson_r'
POINT_SIZE = 6
POINT_ALPHA = 0.75
DPI = 320
PLOT_TAG = 'ICVD_visit12'

def build_output_path(filename: str, out_dir: Path=OUT_DIR) -> str:
    out_dir.mkdir(parents=True, exist_ok=True)
    (stem, ext) = os.path.splitext(filename)
    candidate = out_dir / filename
    if not candidate.exists():
        return str(candidate)
    i = 1
    while True:
        candidate = out_dir / f'{stem}_v{i:02d}{ext}'
        if not candidate.exists():
            return str(candidate)
        i += 1

def load_npy_any(path: Path):
    obj = np.load(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path), allow_pickle=True)
    if isinstance(obj, np.ndarray) and obj.shape == () and (obj.dtype == object):
        return obj.item()
    return obj

def clean_id(x):
    x = pd.Series(x).astype(str)
    x = x.str.replace('.0', '', regex=False)
    x = x.str.strip()
    return x

def to_float_or_nan(x):
    try:
        if x is None:
            return np.nan
        if isinstance(x, (list, tuple, np.ndarray)):
            arr = np.array(x, dtype=object).reshape(-1)
            if len(arr) == 1:
                x = arr[0]
            else:
                return np.nan
        return float(x)
    except Exception:
        return np.nan

def parse_sample_id(sample_key: str):
    s = str(sample_key)
    if '-' in s:
        (pid, visit) = s.split('-', 1)
        return (str(pid), str(visit))
    return (str(s), 'UNK')

def parse_risk_event_time(v):
    risk = np.nan
    event = np.nan
    time = np.nan
    if isinstance(v, dict):
        for k in ['risk', 'pred', 'score', 'rank', 'yhat', 'risk_raw', 'risk_norm', 'prob']:
            if k in v:
                risk = to_float_or_nan(v.get(k))
                if np.isfinite(risk):
                    break
        if not np.isfinite(risk):
            for val in v.values():
                num = to_float_or_nan(val)
                if np.isfinite(num):
                    risk = num
                    break
        for k in ['event', 'event_binary', 'label', 'y', 'outcome', 'case', 'status']:
            if k in v:
                event = to_float_or_nan(v.get(k))
                if np.isfinite(event):
                    break
        for k in ['time', 'tte', 'time_to_event', 'surv_time', 'followup_time', 'duration']:
            if k in v:
                time = to_float_or_nan(v.get(k))
                if np.isfinite(time):
                    break
    elif isinstance(v, (list, tuple, np.ndarray)):
        arr = np.array(v, dtype=object).reshape(-1)
        if len(arr) > 0:
            risk = to_float_or_nan(arr[0])
        if len(arr) > 1:
            event = to_float_or_nan(arr[1])
        if len(arr) > 2:
            time = to_float_or_nan(arr[2])
    else:
        risk = to_float_or_nan(v)
    return (risk, event, time)

def rank_norm(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)
    out = np.full(len(x), np.nan, dtype=float)
    if ok.sum() == 0:
        return out
    r = rankdata(x[ok], method='average')
    if len(r) == 1:
        out[ok] = 0.5
    else:
        out[ok] = (r - 1) / (len(r) - 1)
    return out

def fuse_attn_embedding(emb, attn):
    E = np.asarray(emb, dtype=float)
    A = np.asarray(attn, dtype=float).reshape(-1)
    if E.ndim == 1:
        return E.astype(float)
    if E.ndim != 2:
        raise ValueError(f'Unexpected embedding shape: {E.shape}')
    n_mod = E.shape[0]
    if A.shape[0] != n_mod:
        A = np.ones(n_mod, dtype=float) / max(n_mod, 1)
    else:
        s = A.sum()
        if not np.isfinite(s) or abs(s) < 1e-12:
            A = np.ones(n_mod, dtype=float) / max(n_mod, 1)
        else:
            A = A / s
    return (A[:, None] * E).sum(axis=0)

def load_age_meta(age_path):
    df_meta = pd.read_csv(str(OUTPUT_ROOT / age_path if (OUTPUT_ROOT / age_path).exists() else INPUT_ROOT / age_path), sep='\t', dtype={'ID': str}, low_memory=False)
    keep_cols = ['ID', 'age1992', 'sex']
    missing = [c for c in keep_cols if c not in df_meta.columns]
    if missing:
        raise ValueError(f'Missing columns in AGE_PATH: {missing}')
    df_meta = df_meta[keep_cols].copy()
    df_meta['ID'] = clean_id(df_meta['ID'])
    df_meta['age1992'] = pd.to_numeric(df_meta['age1992'], errors='coerce')
    df_meta['sex'] = pd.to_numeric(df_meta['sex'], errors='coerce')
    print('\nAGE meta before collapsing:')
    print('Rows:', len(df_meta))
    print('Unique IDs:', df_meta['ID'].nunique())
    df_meta = df_meta.sort_values('ID').groupby('ID', as_index=False).agg({'age1992': 'first', 'sex': 'first'})
    print('AGE meta after collapsing:')
    print('Rows:', len(df_meta))
    print('Unique IDs:', df_meta['ID'].nunique())
    return df_meta

def standardize_matrix(X):
    X = np.asarray(X, dtype=float)
    X[~np.isfinite(X)] = np.nan
    col_median = np.nanmedian(X, axis=0)
    col_median[~np.isfinite(col_median)] = 0.0
    inds = np.where(~np.isfinite(X))
    X[inds] = np.take(col_median, inds[1])
    scaler = StandardScaler()
    Xz = scaler.fit_transform(X)
    Xz[~np.isfinite(Xz)] = 0.0
    return Xz

def save_fig(fig, basename):
    pdf_path = build_output_path(f'{basename}.pdf')
    png_path = build_output_path(f'{basename}.png')
    fig.savefig(pdf_path, bbox_inches='tight', transparent=True)
    fig.savefig(png_path, dpi=DPI, bbox_inches='tight', transparent=True)
    print('Saved:', pdf_path)
    print('Saved:', png_path)

def basic_clean_ax(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

def diffusion_map_embedding(X, n_neighbors=30, n_components=15, diffusion_time=1):
    X = np.asarray(X, dtype=float)
    X[~np.isfinite(X)] = 0.0
    n = X.shape[0]
    if n < 5:
        raise ValueError('Too few points for diffusion map.')
    k = min(n_neighbors, n - 1)
    print(f'Building kNN graph: n={n}, k={k}')
    nn = NearestNeighbors(n_neighbors=k + 1, metric='euclidean', n_jobs=-1)
    nn.fit(X)
    (distances, indices) = nn.kneighbors(X)
    distances = distances[:, 1:]
    indices = indices[:, 1:]
    valid_dist = distances[np.isfinite(distances) & (distances > 0)]
    if len(valid_dist) == 0:
        sigma = 1.0
    else:
        sigma = np.median(valid_dist)
    if not np.isfinite(sigma) or sigma <= 1e-12:
        sigma = 1.0
    print(f'Diffusion sigma = {sigma:.6g}')
    weights = np.exp(-distances ** 2 / (2 * sigma ** 2))
    weights[~np.isfinite(weights)] = 0.0
    row = np.repeat(np.arange(n), k)
    col = indices.reshape(-1)
    data = weights.reshape(-1)
    W = sparse.csr_matrix((data, (row, col)), shape=(n, n))
    W = W.maximum(W.T)
    d = np.asarray(W.sum(axis=1)).reshape(-1)
    d[d <= 1e-12] = 1e-12
    d_inv_sqrt = 1.0 / np.sqrt(d)
    D_inv_sqrt = sparse.diags(d_inv_sqrt)
    S = D_inv_sqrt @ W @ D_inv_sqrt
    n_eigs = min(n_components + 1, n - 1)
    print(f'Eigen decomposition: n_eigs={n_eigs}')
    (vals, vecs) = eigsh(S, k=n_eigs, which='LA')
    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    vals_nontrivial = vals[1:n_components + 1]
    vecs_nontrivial = vecs[:, 1:n_components + 1]
    psi = D_inv_sqrt @ vecs_nontrivial
    Z = psi * vals_nontrivial ** diffusion_time
    Z = np.asarray(Z, dtype=float)
    for i in range(Z.shape[1]):
        Z[:, i] = StandardScaler().fit_transform(Z[:, i].reshape(-1, 1)).reshape(-1)
    return (Z, vals)

def pca_embedding(X, n_components=15):
    X = standardize_matrix(X)
    n_components = min(n_components, X.shape[0] - 1, X.shape[1])
    if n_components < 1:
        raise ValueError('Too few samples/features for PCA.')
    pca = PCA(n_components=n_components, random_state=42)
    Z = pca.fit_transform(X)
    for i in range(Z.shape[1]):
        Z[:, i] = StandardScaler().fit_transform(Z[:, i].reshape(-1, 1)).reshape(-1)
    return (Z, pca)

def add_component_columns(meta_df, Z, prefix):
    out = meta_df.copy().reset_index(drop=True)
    for i in range(Z.shape[1]):
        out[f'{prefix}{i + 1}'] = Z[:, i]
    return out

def compute_component_risk_corr(df, risk_col='risk_norm', prefix='DC'):
    comp_cols = [c for c in df.columns if c.startswith(prefix) and c[len(prefix):].isdigit()]
    comp_cols = sorted(comp_cols, key=lambda x: int(x.replace(prefix, '')))
    results = []
    for comp in comp_cols:
        x = pd.to_numeric(df[comp], errors='coerce').values
        y = pd.to_numeric(df[risk_col], errors='coerce').values
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() >= 5:
            (pr, pp) = pearsonr(x[mask], y[mask])
            (sr, sp) = spearmanr(x[mask], y[mask])
        else:
            (pr, pp, sr, sp) = (np.nan, np.nan, np.nan, np.nan)
        results.append({'component': comp, 'component_num': int(comp.replace(prefix, '')), 'n': int(mask.sum()), 'pearson_r': pr, 'pearson_p': pp, 'spearman_r': sr, 'spearman_p': sp, 'abs_pearson_r': abs(pr) if np.isfinite(pr) else np.nan, 'abs_spearman_r': abs(sr) if np.isfinite(sr) else np.nan})
    return pd.DataFrame(results).sort_values('component_num')

def pick_top2_components(corr_df, metric='abs_pearson_r'):
    sub = corr_df.sort_values(metric, ascending=False).reset_index(drop=True)
    if len(sub) == 0:
        raise ValueError('No correlation result found.')
    if len(sub) == 1:
        return (sub.loc[0, 'component'], sub.loc[0, 'component'])
    return (sub.loc[0, 'component'], sub.loc[1, 'component'])

def load_visit12_dataset():
    print('===================================================')
    print('Loading core files')
    print('===================================================')
    print('Embedding:', EMB_PATH)
    print('Attention:', ATTN_PATH)
    print('Risk:', RISK_PATH)
    print('Age meta:', AGE_PATH)
    emb_obj = load_npy_any(EMB_PATH)
    attn_obj = load_npy_any(ATTN_PATH)
    risk_obj = load_npy_any(RISK_PATH)
    if not isinstance(emb_obj, dict):
        raise ValueError('Embedding npy is expected to be dict.')
    if not isinstance(attn_obj, dict):
        raise ValueError('Attention npy is expected to be dict.')
    if not isinstance(risk_obj, dict):
        raise ValueError('Risk npy is expected to be dict.')
    common_keys = sorted(set(emb_obj.keys()) & set(attn_obj.keys()) & set(risk_obj.keys()))
    print('Common aligned keys:', len(common_keys))
    rows = []
    emb_list = []
    attn_list = []
    for k in common_keys:
        emb = np.asarray(emb_obj[k], dtype=float)
        attn = np.asarray(attn_obj[k], dtype=float).reshape(-1)
        if emb.ndim != 2:
            continue
        if emb.shape[0] != len(MODALITY_NAMES):
            continue
        (risk_raw, event_raw, time_raw) = parse_risk_event_time(risk_obj[k])
        (pid, visit) = parse_sample_id(k)
        rows.append({'sample_key': str(k), 'pid': str(pid), 'visit': str(visit), 'risk_raw': risk_raw, 'event_raw': event_raw, 'time_raw': time_raw})
        emb_list.append(emb)
        attn_list.append(attn)
    if len(rows) == 0:
        raise ValueError('No valid aligned samples found.')
    meta = pd.DataFrame(rows)
    emb_arr = np.stack(emb_list, axis=0)
    attn_arr = np.stack(attn_list, axis=0)
    print('\nRaw loaded samples:', len(meta))
    print('Embedding array shape:', emb_arr.shape)
    print('Attention array shape:', attn_arr.shape)
    print('Visit counts before filtering:')
    print(meta['visit'].value_counts(dropna=False).sort_index())
    meta['pid'] = clean_id(meta['pid'])
    meta['visit'] = meta['visit'].astype(str)
    meta['risk_raw'] = pd.to_numeric(meta['risk_raw'], errors='coerce')
    meta['event'] = pd.to_numeric(meta['event_raw'], errors='coerce')
    meta['time'] = pd.to_numeric(meta['time_raw'], errors='coerce')
    meta['event_binary'] = np.nan
    meta.loc[meta['event'] >= 0.5, 'event_binary'] = 1
    meta.loc[meta['event'] < 0.5, 'event_binary'] = 0
    age_meta = load_age_meta(AGE_PATH)
    meta = meta.merge(age_meta, left_on='pid', right_on='ID', how='left')
    if len(meta) != emb_arr.shape[0]:
        raise ValueError(f'Length mismatch after age merge: len(meta)={len(meta)}, emb_arr={emb_arr.shape[0]}')
    meta['age'] = np.nan
    meta.loc[meta['visit'] == '12', 'age'] = meta.loc[meta['visit'] == '12', 'age1992'] + 20
    meta.loc[meta['visit'] == '20', 'age'] = meta.loc[meta['visit'] == '20', 'age1992'] + 28
    visit_mask = (meta['visit'] == TARGET_VISIT).values
    meta = meta.loc[visit_mask].copy().reset_index(drop=True)
    emb_arr = emb_arr[visit_mask, :, :]
    attn_arr = attn_arr[visit_mask, :]
    if len(meta) == 0:
        raise ValueError('No visit12 samples left after filtering.')
    meta['risk_norm'] = rank_norm(meta['risk_raw'].values)
    meta['age'] = pd.to_numeric(meta['age'], errors='coerce')
    meta['sex'] = pd.to_numeric(meta['sex'], errors='coerce')
    meta['event_binary'] = pd.to_numeric(meta['event_binary'], errors='coerce')
    print('\n===================================================')
    print('After visit12 filtering')
    print('===================================================')
    print('N samples:', len(meta))
    print('Embedding shape:', emb_arr.shape)
    print('Attention shape:', attn_arr.shape)
    print('Risk finite:', np.isfinite(meta['risk_norm']).sum())
    print('Age finite:', np.isfinite(meta['age']).sum())
    print('Event finite:', np.isfinite(meta['event_binary']).sum())
    print('\nEvent count:')
    print(meta['event_binary'].value_counts(dropna=False).sort_index())
    meta_path = build_output_path(f'{PLOT_TAG}_sample_meta_visit12.csv')
    meta.to_csv(meta_path, index=False)
    print('Saved:', meta_path)
    return (meta, emb_arr, attn_arr)

def run_modality_reduction(meta, emb_arr, method='diffmap'):
    results = {}
    n_mod = emb_arr.shape[1]
    if n_mod != len(MODALITY_NAMES):
        raise ValueError(f'Embedding modality dim = {n_mod}, but MODALITY_NAMES has {len(MODALITY_NAMES)} names.')
    if method == 'diffmap':
        prefix = 'DC'
        tag = 'DiffMap'
    elif method == 'pca':
        prefix = 'PC'
        tag = 'PCA'
    else:
        raise ValueError("method must be 'diffmap' or 'pca'.")
    for (m, mod_name) in enumerate(MODALITY_NAMES):
        print('\n===================================================')
        print(f'Running {tag}: {mod_name}')
        print('===================================================')
        X = emb_arr[:, m, :]
        X = standardize_matrix(X)
        if method == 'diffmap':
            (Z, model_obj) = diffusion_map_embedding(X, n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS, diffusion_time=DIFFUSION_TIME)
        else:
            (Z, model_obj) = pca_embedding(X, n_components=N_COMPONENTS)
        df_comp = add_component_columns(meta, Z, prefix=prefix)
        corr_df = compute_component_risk_corr(df_comp, risk_col='risk_norm', prefix=prefix)
        (xcomp, ycomp) = pick_top2_components(corr_df, metric=TOP_AXIS_METRIC)
        corr_path = build_output_path(f'{PLOT_TAG}_{method}_{mod_name}_{prefix}_risk_corr.csv')
        coords_path = build_output_path(f'{PLOT_TAG}_{method}_{mod_name}_coords.csv')
        corr_df.to_csv(corr_path, index=False)
        df_comp.to_csv(coords_path, index=False)
        print(f'Best two {prefix}s for {mod_name}: {xcomp}, {ycomp}')
        print(corr_df.sort_values(TOP_AXIS_METRIC, ascending=False).head(5))
        print('Saved:', corr_path)
        print('Saved:', coords_path)
        results[mod_name] = {'method': method, 'prefix': prefix, 'X': X, 'Z': Z, 'model': model_obj, 'df': df_comp, 'corr': corr_df, 'xcomp': xcomp, 'ycomp': ycomp}
    return results

def run_fused_reduction(meta, emb_arr, attn_arr, method='pca'):
    if method == 'diffmap':
        prefix = 'DC'
        tag = 'DiffMap'
    elif method == 'pca':
        prefix = 'PC'
        tag = 'PCA'
    else:
        raise ValueError("method must be 'diffmap' or 'pca'.")
    print('\n===================================================')
    print(f'Running fused attention-weighted {tag}')
    print('===================================================')
    fused_list = []
    for i in range(len(meta)):
        fused = fuse_attn_embedding(emb_arr[i], attn_arr[i])
        fused_list.append(fused)
    X_fused = np.vstack(fused_list).astype(float)
    X_fused = standardize_matrix(X_fused)
    if method == 'diffmap':
        (Z, model_obj) = diffusion_map_embedding(X_fused, n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS, diffusion_time=DIFFUSION_TIME)
    else:
        (Z, model_obj) = pca_embedding(X_fused, n_components=N_COMPONENTS)
    df_comp = add_component_columns(meta, Z, prefix=prefix)
    corr_df = compute_component_risk_corr(df_comp, risk_col='risk_norm', prefix=prefix)
    (xcomp, ycomp) = pick_top2_components(corr_df, metric=TOP_AXIS_METRIC)
    corr_path = build_output_path(f'{PLOT_TAG}_{method}_FUSED_{prefix}_risk_corr.csv')
    coords_path = build_output_path(f'{PLOT_TAG}_{method}_FUSED_coords.csv')
    corr_df.to_csv(corr_path, index=False)
    df_comp.to_csv(coords_path, index=False)
    print(f'Best two {prefix}s for FUSED: {xcomp}, {ycomp}')
    print(corr_df.sort_values(TOP_AXIS_METRIC, ascending=False).head(5))
    print('Saved:', corr_path)
    print('Saved:', coords_path)
    return {'method': method, 'prefix': prefix, 'X': X_fused, 'Z': Z, 'model': model_obj, 'df': df_comp, 'corr': corr_df, 'xcomp': xcomp, 'ycomp': ycomp}

def plot_modality_fourpanel(results, method='pca'):
    order = ['Fundus', 'SNP', 'Soma', 'ECG']
    if method == 'diffmap':
        label = 'DiffMap'
    elif method == 'pca':
        label = 'PCA'
    else:
        label = method
    (fig, axes) = plt.subplots(2, 2, figsize=(10.5, 8.5))
    axes = axes.flatten()
    all_risk = []
    for mod in order:
        df = results[mod]['df']
        all_risk.extend(pd.to_numeric(df['risk_norm'], errors='coerce').values.tolist())
    all_risk = np.array(all_risk, dtype=float)
    valid_risk = all_risk[np.isfinite(all_risk)]
    if len(valid_risk) > 0:
        (vmin, vmax) = np.nanpercentile(valid_risk, [2, 98])
    else:
        (vmin, vmax) = (0.0, 1.0)
    sc = None
    for (ax, mod) in zip(axes, order):
        obj = results[mod]
        df = obj['df']
        corr_df = obj['corr']
        xcomp = obj['xcomp']
        ycomp = obj['ycomp']
        sub = corr_df.set_index('component')
        rx = sub.loc[xcomp, 'pearson_r'] if xcomp in sub.index else np.nan
        ry = sub.loc[ycomp, 'pearson_r'] if ycomp in sub.index else np.nan
        x = pd.to_numeric(df[xcomp], errors='coerce').values
        y = pd.to_numeric(df[ycomp], errors='coerce').values
        c = pd.to_numeric(df['risk_norm'], errors='coerce').values
        mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(c)
        sc = ax.scatter(x[mask], y[mask], c=np.clip(c[mask], vmin, vmax), cmap='magma', s=POINT_SIZE, alpha=POINT_ALPHA, vmin=vmin, vmax=vmax, linewidths=0)
        ax.set_title(f'{mod}\n{xcomp} (r={rx:.3f}) vs {ycomp} (r={ry:.3f})', fontsize=11)
        basic_clean_ax(ax)
    cbar = fig.colorbar(sc, ax=axes.tolist(), fraction=0.03, pad=0.02)
    cbar.set_label('Normalized risk')
    plt.tight_layout()
    save_fig(fig, f'{PLOT_TAG}_{method}_modalities_fourpanel_top_risk_axes')
    plt.close(fig)

def plot_fused_fourpanel(fused_result, method='pca'):
    df = fused_result['df']
    corr_df = fused_result['corr']
    xcomp = fused_result['xcomp']
    ycomp = fused_result['ycomp']
    if method == 'diffmap':
        label = 'Fused DiffMap'
    elif method == 'pca':
        label = 'Fused PCA'
    else:
        label = f'Fused {method}'
    sub = corr_df.set_index('component')
    rx = sub.loc[xcomp, 'pearson_r'] if xcomp in sub.index else np.nan
    ry = sub.loc[ycomp, 'pearson_r'] if ycomp in sub.index else np.nan
    x = pd.to_numeric(df[xcomp], errors='coerce').values
    y = pd.to_numeric(df[ycomp], errors='coerce').values
    risk = pd.to_numeric(df['risk_norm'], errors='coerce').values
    age = pd.to_numeric(df['age'], errors='coerce').values
    event = pd.to_numeric(df['event_binary'], errors='coerce').values
    (fig, axes) = plt.subplots(2, 2, figsize=(10.5, 8.5))
    ax_plain = axes[0, 0]
    ax_risk = axes[0, 1]
    ax_age = axes[1, 0]
    ax_event = axes[1, 1]
    mask_plain = np.isfinite(x) & np.isfinite(y)
    ax_plain.scatter(x[mask_plain], y[mask_plain], s=POINT_SIZE, alpha=0.55, color='gray', linewidths=0)
    ax_plain.set_title(f'{label}\n{xcomp} (r={rx:.3f}) vs {ycomp} (r={ry:.3f})')
    basic_clean_ax(ax_plain)
    mask_risk = np.isfinite(x) & np.isfinite(y) & np.isfinite(risk)
    if mask_risk.sum() > 0:
        (vmin_r, vmax_r) = np.nanpercentile(risk[mask_risk], [2, 98])
        sc1 = ax_risk.scatter(x[mask_risk], y[mask_risk], c=np.clip(risk[mask_risk], vmin_r, vmax_r), cmap='magma', s=POINT_SIZE, alpha=POINT_ALPHA, vmin=vmin_r, vmax=vmax_r, linewidths=0)
        fig.colorbar(sc1, ax=ax_risk, fraction=0.046).set_label('Normalized risk')
    ax_risk.set_title('Risk')
    basic_clean_ax(ax_risk)
    mask_age = np.isfinite(x) & np.isfinite(y) & np.isfinite(age)
    if mask_age.sum() > 0:
        sc2 = ax_age.scatter(x[mask_age], y[mask_age], c=age[mask_age], cmap='viridis', s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0)
        fig.colorbar(sc2, ax=ax_age, fraction=0.046).set_label('Age')
    ax_age.set_title('Age')
    basic_clean_ax(ax_age)
    mask0 = np.isfinite(x) & np.isfinite(y) & (event == 0)
    mask1 = np.isfinite(x) & np.isfinite(y) & (event == 1)
    maskm = np.isfinite(x) & np.isfinite(y) & ~np.isfinite(event)
    if maskm.sum() > 0:
        ax_event.scatter(x[maskm], y[maskm], s=POINT_SIZE, alpha=0.2, color='lightgray', linewidths=0)
    if mask0.sum() > 0:
        ax_event.scatter(x[mask0], y[mask0], s=POINT_SIZE, alpha=0.5, color='#4C72B0', linewidths=0)
    if mask1.sum() > 0:
        ax_event.scatter(x[mask1], y[mask1], s=POINT_SIZE + 2, alpha=0.9, color='#C44E52', linewidths=0)
    ax_event.set_title('Event')
    basic_clean_ax(ax_event)
    event_legend = [Line2D([0], [0], marker='o', color='w', label='No event', markerfacecolor='#4C72B0', markersize=6), Line2D([0], [0], marker='o', color='w', label='Event', markerfacecolor='#C44E52', markersize=6)]
    ax_event.legend(handles=event_legend, frameon=False, loc='best')
    plt.tight_layout()
    save_fig(fig, f'{PLOT_TAG}_{method}_FUSED_fourpanel_plain_risk_age_event')
    plt.close(fig)

def plot_pearson_curve(results, method='diffmap'):
    if method == 'diffmap':
        x_label = 'Diffusion component'
        title = 'DiffMap component-risk Pearson correlation'
        out_name = f'{PLOT_TAG}_diffmap_modality_DC1_DC15_pearson_curve'
    elif method == 'pca':
        x_label = 'Principal component'
        title = 'PCA component-risk Pearson correlation'
        out_name = f'{PLOT_TAG}_pca_modality_PC1_PC15_pearson_curve'
    else:
        x_label = 'Component'
        title = f'{method} component-risk Pearson correlation'
        out_name = f'{PLOT_TAG}_{method}_modality_component_pearson_curve'
    (fig, ax) = plt.subplots(figsize=(7.4, 4.6))
    for mod in MODALITY_NAMES:
        corr = results[mod]['corr'].copy()
        corr = corr.sort_values('component_num')
        ax.plot(corr['component_num'], corr['pearson_r'], marker='o', linewidth=1.8, label=mod)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel(x_label)
    ax.set_ylabel('Pearson r with normalized risk')
    ax.set_title(title)
    ax.set_xticks(range(1, N_COMPONENTS + 1))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, fontsize=9, loc='best')
    plt.tight_layout()
    save_fig(fig, out_name)
    plt.close(fig)

def save_combined_corr_table(diffmap_results, pca_results):
    rows = []
    for (method, results) in [('diffmap', diffmap_results), ('pca', pca_results)]:
        for mod in MODALITY_NAMES:
            corr = results[mod]['corr'].copy()
            corr['method'] = method
            corr['modality'] = mod
            rows.append(corr)
    out = pd.concat(rows, axis=0, ignore_index=True)
    out = out[['method', 'modality', 'component', 'component_num', 'n', 'pearson_r', 'pearson_p', 'spearman_r', 'spearman_p', 'abs_pearson_r', 'abs_spearman_r']]
    out_path = build_output_path(f'{PLOT_TAG}_diffmap_pca_modality_component_risk_corr_all.csv')
    out.to_csv(out_path, index=False)
    print('Saved combined correlation table:', out_path)
    return out

def main():
    (meta, emb_arr, attn_arr) = load_visit12_dataset()
    print('\n===================================================')
    print('Running modality-level DiffMap')
    print('===================================================')
    diffmap_results = run_modality_reduction(meta, emb_arr, method='diffmap')
    plot_pearson_curve(diffmap_results, method='diffmap')
    print('\n===================================================')
    print('Running modality-level PCA')
    print('===================================================')
    pca_results = run_modality_reduction(meta, emb_arr, method='pca')
    plot_modality_fourpanel(pca_results, method='pca')
    plot_pearson_curve(pca_results, method='pca')
    print('\n===================================================')
    print('Running fused attention-weighted PCA')
    print('===================================================')
    fused_pca_result = run_fused_reduction(meta, emb_arr, attn_arr, method='pca')
    plot_fused_fourpanel(fused_pca_result, method='pca')
    print('\n===================================================')
    print('Running fused attention-weighted DiffMap')
    print('===================================================')
    fused_diffmap_result = run_fused_reduction(meta, emb_arr, attn_arr, method='diffmap')
    plot_fused_fourpanel(fused_diffmap_result, method='diffmap')
    save_combined_corr_table(diffmap_results, pca_results)
    print('\n===================================================')
    print('Done.')
    print('All outputs saved to:')
    print(OUT_DIR)
    print('===================================================')
if __name__ == '__main__':
    main()


### Figure 4f — Principal-component correlations with risk


In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import rankdata, pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
warnings.filterwarnings('ignore')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
BASE_DIR = Path('fig3/code_save_6_7')
OUT_DIR = Path('fig4_final/embeddingv4')
OUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_PATH = BASE_DIR / 'sample_embeddings_ICVD_test.npy'
RISK_PATH = BASE_DIR / 'sample_risk_ICVD_test.npy'
TARGET_VISIT = '12'
MODALITY_NAMES = ['Fundus', 'SNP', 'Soma', 'ECG']
PLOT_LABELS = ['Fundus', 'SNP', 'Soma', 'ECG']
COLOR_MAP = {'Fundus': '#6F6F6F', 'SNP': '#A07A78', 'Soma': '#4F8080', 'ECG': '#A67D5D'}
COLOR_MAP = {'Fundus': '#778368', 'ECG': '#658a9a', 'Soma': '#9382a0', 'SNP': '#9a6c6c'}
N_COMPONENTS = 10
FIG_W_MM = 50
FIG_H_MM = 50
MM_TO_INCH = 1 / 25.4
DPI = 320
PLOT_TAG = 'ICVD_visit12_PCA_PC1_PC10_risk_corr'
OUT_PDF = OUT_DIR / f'{PLOT_TAG}_smooth_curve_v02.pdf'

def load_npy_any(path: Path):
    obj = np.load(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path), allow_pickle=True)
    if isinstance(obj, np.ndarray) and obj.shape == () and (obj.dtype == object):
        return obj.item()
    return obj

def to_float_or_nan(x):
    try:
        if x is None:
            return np.nan
        if isinstance(x, (list, tuple, np.ndarray)):
            arr = np.array(x, dtype=object).reshape(-1)
            if len(arr) == 1:
                x = arr[0]
            else:
                return np.nan
        return float(x)
    except Exception:
        return np.nan

def parse_sample_id(sample_key):
    s = str(sample_key)
    if '-' in s:
        (pid, visit) = s.split('-', 1)
        return (str(pid), str(visit))
    return (str(s), 'UNK')

def parse_risk_event_time(v):
    risk = np.nan
    event = np.nan
    time = np.nan
    if isinstance(v, dict):
        for k in ['risk', 'pred', 'score', 'rank', 'yhat', 'risk_raw', 'risk_norm', 'prob']:
            if k in v:
                risk = to_float_or_nan(v.get(k))
                if np.isfinite(risk):
                    break
        if not np.isfinite(risk):
            for val in v.values():
                num = to_float_or_nan(val)
                if np.isfinite(num):
                    risk = num
                    break
        for k in ['event', 'event_binary', 'label', 'y', 'outcome', 'case', 'status']:
            if k in v:
                event = to_float_or_nan(v.get(k))
                if np.isfinite(event):
                    break
        for k in ['time', 'tte', 'time_to_event', 'surv_time', 'followup_time', 'duration']:
            if k in v:
                time = to_float_or_nan(v.get(k))
                if np.isfinite(time):
                    break
    elif isinstance(v, (list, tuple, np.ndarray)):
        arr = np.array(v, dtype=object).reshape(-1)
        if len(arr) > 0:
            risk = to_float_or_nan(arr[0])
        if len(arr) > 1:
            event = to_float_or_nan(arr[1])
        if len(arr) > 2:
            time = to_float_or_nan(arr[2])
    else:
        risk = to_float_or_nan(v)
    return (risk, event, time)

def rank_norm(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)
    out = np.full(len(x), np.nan, dtype=float)
    if ok.sum() == 0:
        return out
    r = rankdata(x[ok], method='average')
    if len(r) == 1:
        out[ok] = 0.5
    else:
        out[ok] = (r - 1) / (len(r) - 1)
    return out

def standardize_matrix(X):
    X = np.asarray(X, dtype=float)
    X[~np.isfinite(X)] = np.nan
    col_median = np.nanmedian(X, axis=0)
    col_median[~np.isfinite(col_median)] = 0.0
    inds = np.where(~np.isfinite(X))
    X[inds] = np.take(col_median, inds[1])
    Xz = StandardScaler().fit_transform(X)
    Xz[~np.isfinite(Xz)] = 0.0
    return Xz

def pc_to_compressed_x(pc):
    pc = np.asarray(pc, dtype=float)
    out = np.zeros_like(pc, dtype=float)
    mask1 = pc <= 1
    mask2 = pc >= 2
    out[mask1] = 0.0
    out[mask2] = 0.5 + 0.5 * np.log(pc[mask2] / 2.0) / np.log(10.0 / 2.0)
    return out

def load_visit12_data():
    print('===================================================')
    print('Loading data')
    print('===================================================')
    print('Embedding:', EMB_PATH)
    print('Risk:', RISK_PATH)
    emb_obj = load_npy_any(EMB_PATH)
    risk_obj = load_npy_any(RISK_PATH)
    if not isinstance(emb_obj, dict):
        raise ValueError('Expected embedding npy to be dict.')
    if not isinstance(risk_obj, dict):
        raise ValueError('Expected risk npy to be dict.')
    common_keys = sorted(set(emb_obj.keys()) & set(risk_obj.keys()))
    print('Common keys:', len(common_keys))
    rows = []
    emb_list = []
    for k in common_keys:
        emb = np.asarray(emb_obj[k], dtype=float)
        if emb.ndim != 2:
            continue
        if emb.shape[0] != len(MODALITY_NAMES):
            continue
        (risk_raw, event_raw, time_raw) = parse_risk_event_time(risk_obj[k])
        (pid, visit) = parse_sample_id(k)
        rows.append({'sample_key': str(k), 'pid': str(pid), 'visit': str(visit), 'risk_raw': risk_raw, 'event_raw': event_raw, 'time_raw': time_raw})
        emb_list.append(emb)
    if len(rows) == 0:
        raise ValueError('No valid samples found.')
    meta = pd.DataFrame(rows)
    emb_arr = np.stack(emb_list, axis=0)
    print('Raw embedding shape:', emb_arr.shape)
    print('Visit counts before filtering:')
    print(meta['visit'].value_counts(dropna=False).sort_index())
    meta['visit'] = meta['visit'].astype(str)
    meta['risk_raw'] = pd.to_numeric(meta['risk_raw'], errors='coerce')
    visit_mask = (meta['visit'] == TARGET_VISIT).values
    meta = meta.loc[visit_mask].copy().reset_index(drop=True)
    emb_arr = emb_arr[visit_mask, :, :]
    if len(meta) == 0:
        raise ValueError(f'No samples left after filtering visit == {TARGET_VISIT}')
    meta['risk_norm'] = rank_norm(meta['risk_raw'].values)
    print('\nAfter visit12 filtering')
    print('N samples:', len(meta))
    print('Embedding shape:', emb_arr.shape)
    print('Risk finite:', np.isfinite(meta['risk_norm']).sum())
    return (meta, emb_arr)

def compute_pca_pc_risk_corr(meta, emb_arr):
    all_rows = []
    risk = pd.to_numeric(meta['risk_norm'], errors='coerce').values
    for (m, mod_name) in enumerate(MODALITY_NAMES):
        label = PLOT_LABELS[m]
        print('\n===================================================')
        print(f'Running PCA for {mod_name} -> {label}')
        print('===================================================')
        X = emb_arr[:, m, :]
        X = standardize_matrix(X)
        n_components = min(N_COMPONENTS, X.shape[0] - 1, X.shape[1])
        pca = PCA(n_components=n_components, random_state=42)
        Z = pca.fit_transform(X)
        for pc_idx in range(n_components):
            pc = Z[:, pc_idx]
            mask = np.isfinite(pc) & np.isfinite(risk)
            if mask.sum() >= 5:
                (r, p) = pearsonr(pc[mask], risk[mask])
            else:
                (r, p) = (np.nan, np.nan)
            all_rows.append({'modality_raw': mod_name, 'plot_label': label, 'pc': f'PC{pc_idx + 1}', 'pc_num': pc_idx + 1, 'n': int(mask.sum()), 'pearson_r': r, 'pearson_p': p, 'explained_variance_ratio': pca.explained_variance_ratio_[pc_idx]})
    corr_df = pd.DataFrame(all_rows)
    out_csv = OUT_DIR / f'{PLOT_TAG}_pearson_corr_v02.csv'
    corr_df.to_csv(out_csv, index=False)
    print('\nSaved:', out_csv)
    return corr_df

def plot_pc_corr_polyline_compressed_x(corr_df):
    (fig, ax) = plt.subplots(figsize=(FIG_W_MM * MM_TO_INCH, FIG_H_MM * MM_TO_INCH))
    for label in PLOT_LABELS:
        sub = corr_df[corr_df['plot_label'] == label].copy()
        sub = sub.sort_values('pc_num')
        pc_num = sub['pc_num'].values.astype(float)
        y = sub['pearson_r'].values.astype(float)
        mask = np.isfinite(pc_num) & np.isfinite(y)
        pc_num = pc_num[mask]
        y = y[mask]
        x = pc_to_compressed_x(pc_num)
        color = COLOR_MAP[label]
        ax.plot(x, y, color=color, linewidth=1.4, marker='o', markersize=2.6, markerfacecolor=color, markeredgecolor=color, markeredgewidth=0, label=label)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylim(-0.25, 0.6)
    pc_ticks = np.arange(1, N_COMPONENTS + 1)
    tick_pos = pc_to_compressed_x(pc_ticks)
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([f'PC{i}' for i in pc_ticks], rotation=45, ha='right', fontsize=5.5)
    ax.set_xlabel('')
    ax.set_ylabel('Pearson r', fontsize=7)
    ax.tick_params(axis='y', labelsize=6, width=0.6, length=2)
    ax.tick_params(axis='x', width=0.6, length=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.6)
    ax.spines['bottom'].set_linewidth(0.6)
    ax.legend(frameon=False, fontsize=5.8, loc='best', handlelength=1.2, borderaxespad=0.2)
    plt.tight_layout(pad=0.35)
    fig.savefig(OUT_PDF, bbox_inches='tight', transparent=True)
    print('Saved:', OUT_PDF)
    plt.close(fig)

def main():
    (meta, emb_arr) = load_visit12_data()
    corr_df = compute_pca_pc_risk_corr(meta, emb_arr)
    plot_pc_corr_polyline_compressed_x(corr_df)
    print('\nDone.')
    print('Output:', OUT_PDF)
if __name__ == '__main__':
    main()
